# Volume Filing & Merit Inflation Analysis

**Requirement 7** (Volume Filing — Merit Inflation) × **Logic 5** (Predictable Missed Deadlines)

Primary table: `idre.idre_silver.oon_emergency_nonemergency_fee_join` (Q1 + Q2 combined)

## Column Mappings Applied
| Agent Prompt Column | Actual Column |
|---|---|
| `inferred_idre` | `idre_name` |
| `prevailing_party_offer_pct_qpa` | `Prevailing_Party_Offer_as_of_QPA` |
| `provider_facility_offer_pct_qpa` | `Provider_Facility_Offer_as_of_QPA` |
| `puf_quarter` | `quarter` |

All thresholds are computed dynamically from the data.

In [0]:
%sql
-- ============================================================
-- Block 1a — The Volume Landscape (by Provider Email Domain)
-- ============================================================
-- Which corporate families dominate dispute volume, what fraction
-- of national disputes do they represent, and how do their merit
-- win rates and default win rates compare to the national baseline?

WITH national_total AS (
    SELECT COUNT(DISTINCT Dispute_Number) AS national_disputes
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
),
provider_stats AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%'
             AND Default_Decision != 'Yes'
            THEN Dispute_Number END) AS merit_wins,
        COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes'
             AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) AS default_wins
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
)
SELECT
    ps.Provider_Email_Domain,
    ps.total_disputes,
    ps.merit_wins,
    ps.default_wins,
    ps.merit_wins + ps.default_wins AS total_wins,
    ROUND(100.0 * ps.merit_wins / NULLIF(ps.total_disputes, 0), 1) AS merit_win_rate,
    ROUND(100.0 * ps.default_wins / NULLIF(ps.total_disputes, 0), 1) AS default_win_rate,
    ROUND(100.0 * (ps.merit_wins + ps.default_wins) / NULLIF(ps.total_disputes, 0), 1) AS total_win_rate,
    ROUND(100.0 * ps.total_disputes / nt.national_disputes, 2) AS national_share_pct,
    CASE
        WHEN 100.0 * ps.total_disputes / nt.national_disputes > 1.0
          OR ps.total_disputes > 5000
        THEN TRUE ELSE FALSE
    END AS flagged,
    CASE
        WHEN 100.0 * ps.total_disputes / nt.national_disputes > 1.0
         AND ps.total_disputes > 5000
        THEN CONCAT('National share ', ROUND(100.0 * ps.total_disputes / nt.national_disputes, 2), '% with ', ps.total_disputes, ' disputes')
        WHEN 100.0 * ps.total_disputes / nt.national_disputes > 1.0
        THEN CONCAT('National share ', ROUND(100.0 * ps.total_disputes / nt.national_disputes, 2), '%')
        WHEN ps.total_disputes > 5000
        THEN CONCAT(ps.total_disputes, ' disputes exceeds 5K threshold')
        ELSE NULL
    END AS flag_reason
FROM provider_stats ps
CROSS JOIN national_total nt
ORDER BY ps.total_disputes DESC

Provider_Email_Domain,total_disputes,merit_wins,default_wins,total_wins,merit_win_rate,default_win_rate,total_win_rate,national_share_pct,flagged,flag_reason
halomd.com,180706,111746,46029,157775,61.8,25.5,87.3,17.07,true,National share 17.07% with 180706 disputes
teamhealth.com,168799,146858,12996,159854,87.0,7.7,94.7,15.95,true,National share 15.95% with 168799 disputes
saparm.com,83824,68621,9786,78407,81.9,11.7,93.5,7.92,true,National share 7.92% with 83824 disputes
agshealth.com,50422,32838,10718,43556,65.1,21.3,86.4,4.76,true,National share 4.76% with 50422 disputes
scp-health.com,47223,31779,10594,42373,67.3,22.4,89.7,4.46,true,National share 4.46% with 47223 disputes
fam-llc.com,38045,14890,16398,31288,39.1,43.1,82.2,3.59,true,National share 3.59% with 38045 disputes
orthomedstaffing.com,30925,19840,6629,26469,64.2,21.4,85.6,2.92,true,National share 2.92% with 30925 disputes
qmacsmso.com,30581,12551,12083,24634,41.0,39.5,80.6,2.89,true,National share 2.89% with 30581 disputes
radpmg.com,25955,22690,2303,24993,87.4,8.9,96.3,2.45,true,National share 2.45% with 25955 disputes
sonoranrm.com,25889,20338,3388,23726,78.6,13.1,91.6,2.45,true,National share 2.45% with 25889 disputes


In [0]:
%sql
-- ============================================================
-- Block 1b — Volume Landscape Fallback (NULL/empty domains)
-- ============================================================
-- Repeats Block 1 grouping by Provider_Facility_Group_Name
-- for rows where Provider_Email_Domain is NULL or empty.

WITH national_total AS (
    SELECT COUNT(DISTINCT Dispute_Number) AS national_disputes
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
),
facility_stats AS (
    SELECT
        Provider_Facility_Group_Name,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%'
             AND Default_Decision != 'Yes'
            THEN Dispute_Number END) AS merit_wins,
        COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes'
             AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) AS default_wins
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Provider_Email_Domain IS NULL
       OR TRIM(Provider_Email_Domain) = ''
    GROUP BY Provider_Facility_Group_Name
)
SELECT
    fs.Provider_Facility_Group_Name,
    fs.total_disputes,
    fs.merit_wins,
    fs.default_wins,
    fs.merit_wins + fs.default_wins AS total_wins,
    ROUND(100.0 * fs.merit_wins / NULLIF(fs.total_disputes, 0), 1) AS merit_win_rate,
    ROUND(100.0 * fs.default_wins / NULLIF(fs.total_disputes, 0), 1) AS default_win_rate,
    ROUND(100.0 * (fs.merit_wins + fs.default_wins) / NULLIF(fs.total_disputes, 0), 1) AS total_win_rate,
    ROUND(100.0 * fs.total_disputes / nt.national_disputes, 2) AS national_share_pct,
    CASE
        WHEN 100.0 * fs.total_disputes / nt.national_disputes > 1.0
          OR fs.total_disputes > 5000
        THEN TRUE ELSE FALSE
    END AS flagged,
    CASE
        WHEN fs.total_disputes > 5000
        THEN CONCAT(fs.total_disputes, ' disputes (NULL domain) exceeds 5K threshold')
        WHEN 100.0 * fs.total_disputes / nt.national_disputes > 1.0
        THEN CONCAT('National share ', ROUND(100.0 * fs.total_disputes / nt.national_disputes, 2), '% (NULL domain)')
        ELSE NULL
    END AS flag_reason
FROM facility_stats fs
CROSS JOIN national_total nt
ORDER BY fs.total_disputes DESC

Provider_Facility_Group_Name,total_disputes,merit_wins,default_wins,total_wins,merit_win_rate,default_win_rate,total_win_rate,national_share_pct,flagged,flag_reason


In [0]:
%sql
-- ============================================================
-- Block 2 — The Financial Extraction Model (Merit Track)
-- ============================================================
-- Total implied financial extraction for each high-volume filer.
-- Line-item level (dli_number), merit wins only.
-- Prevailing_Party_Offer_as_of_QPA = prevailing_party_offer_pct_qpa
-- Provider_Facility_Offer_as_of_QPA = provider_facility_offer_pct_qpa

WITH merit_dlis AS (
    SELECT
        Provider_Email_Domain,
        DLI_Number,
        TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE) AS award_pct_qpa
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Payment_Determination_Outcome ILIKE '%provider%'
      AND Default_Decision != 'Yes'
      AND Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
),
extraction AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DLI_Number) AS merit_win_dli_count,
        ROUND(PERCENTILE(award_pct_qpa, 0.5), 2) AS median_award_pct_qpa,
        ROUND(PERCENTILE(award_pct_qpa, 0.9), 2) AS p90_award_pct_qpa,
        ROUND(AVG(award_pct_qpa), 2) AS avg_award_pct_qpa
    FROM merit_dlis
    WHERE award_pct_qpa IS NOT NULL
    GROUP BY Provider_Email_Domain
),
block1_stats AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%'
             AND Default_Decision != 'Yes'
            THEN Dispute_Number END)
            / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS merit_win_rate
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
)
SELECT
    e.Provider_Email_Domain,
    b.total_disputes,
    b.merit_win_rate,
    e.merit_win_dli_count,
    e.median_award_pct_qpa,
    e.p90_award_pct_qpa,
    e.avg_award_pct_qpa,
    ROUND(e.median_award_pct_qpa * e.merit_win_dli_count / 100.0, 0) AS extraction_index,
    CASE
        WHEN e.median_award_pct_qpa > 500 THEN TRUE
        ELSE FALSE
    END AS flagged,
    CASE
        WHEN e.median_award_pct_qpa > 500
        THEN CONCAT('Median award ', e.median_award_pct_qpa, '% QPA on ', e.merit_win_dli_count, ' winning DLIs. Extraction index: ', ROUND(e.median_award_pct_qpa * e.merit_win_dli_count / 100.0, 0))
        ELSE NULL
    END AS flag_reason
FROM extraction e
JOIN block1_stats b ON e.Provider_Email_Domain = b.Provider_Email_Domain
ORDER BY e.median_award_pct_qpa * e.merit_win_dli_count / 100.0 DESC

Provider_Email_Domain,total_disputes,merit_win_rate,merit_win_dli_count,median_award_pct_qpa,p90_award_pct_qpa,avg_award_pct_qpa,extraction_index,flagged,flag_reason
erkaty.com,186,36.0,146,NaN,NaN,NaN,NaN,true,Median award NaN% QPA on 146 winning DLIs. Extraction index: NaN
adrightmedicalbilling.com,1,100.0,11,NaN,NaN,NaN,NaN,true,Median award NaN% QPA on 11 winning DLIs. Extraction index: NaN
shands.ufl.edu,142,71.1,269,NaN,NaN,NaN,NaN,true,Median award NaN% QPA on 269 winning DLIs. Extraction index: NaN
wellstar.org,887,44.8,3016,NaN,NaN,NaN,NaN,true,Median award NaN% QPA on 3016 winning DLIs. Extraction index: NaN
altushealthsystem.com,9379,30.5,4033,NaN,NaN,NaN,NaN,true,Median award NaN% QPA on 4033 winning DLIs. Extraction index: NaN
syntechhealth.com,742,27.0,830,NaN,NaN,NaN,NaN,true,Median award NaN% QPA on 830 winning DLIs. Extraction index: NaN
neighborshealth.com,802,33.9,506,NaN,NaN,NaN,NaN,true,Median award NaN% QPA on 506 winning DLIs. Extraction index: NaN
erevenuebilling.com,6210,56.4,4989,NaN,NaN,NaN,NaN,true,Median award NaN% QPA on 4989 winning DLIs. Extraction index: NaN
memorialvillageer.com,1894,51.3,3621,NaN,NaN,NaN,NaN,true,Median award NaN% QPA on 3621 winning DLIs. Extraction index: NaN
rightmedicalbilling.com,4621,41.4,1841,NaN,NaN,NaN,NaN,true,Median award NaN% QPA on 1841 winning DLIs. Extraction index: NaN


In [0]:
%sql
-- ============================================================
-- Block 3 — Plan Targeting: Was the Volume Deliberate?
-- ============================================================
-- For each high-volume provider, which health plans do they file
-- against most heavily, and do those plans have abnormally high
-- default loss rates against that specific provider?

WITH provider_volume AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS total_disputes
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
    HAVING COUNT(DISTINCT Dispute_Number) > 1000
),
provider_plan AS (
    SELECT
        t.Provider_Email_Domain,
        t.Health_Plan_Issuer_Name,
        COUNT(DISTINCT t.Dispute_Number) AS disputes_filed,
        COUNT(DISTINCT CASE
            WHEN t.Default_Decision = 'Yes'
             AND t.Payment_Determination_Outcome ILIKE '%provider%'
            THEN t.Dispute_Number END) AS default_wins_vs_plan
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join t
    INNER JOIN provider_volume pv ON t.Provider_Email_Domain = pv.Provider_Email_Domain
    GROUP BY t.Provider_Email_Domain, t.Health_Plan_Issuer_Name
),
with_plan_totals AS (
    SELECT
        pp.*,
        ROUND(100.0 * pp.default_wins_vs_plan / NULLIF(pp.disputes_filed, 0), 1) AS default_rate_vs_plan,
        SUM(pp.default_wins_vs_plan) OVER (PARTITION BY pp.Health_Plan_Issuer_Name) AS plan_total_default_losses,
        ROUND(100.0 * pp.default_wins_vs_plan
            / NULLIF(SUM(pp.default_wins_vs_plan) OVER (PARTITION BY pp.Health_Plan_Issuer_Name), 0), 1) AS provider_share_of_plan_defaults
    FROM provider_plan pp
)
SELECT
    Provider_Email_Domain,
    Health_Plan_Issuer_Name,
    disputes_filed,
    default_wins_vs_plan,
    default_rate_vs_plan,
    plan_total_default_losses,
    provider_share_of_plan_defaults,
    CASE
        WHEN default_rate_vs_plan > 40
         AND provider_share_of_plan_defaults > 20
        THEN TRUE ELSE FALSE
    END AS flagged,
    CASE
        WHEN default_rate_vs_plan > 40
         AND provider_share_of_plan_defaults > 20
        THEN CONCAT('Default rate ', default_rate_vs_plan, '% vs plan, ',
                     provider_share_of_plan_defaults, '% of plan total defaults')
        ELSE NULL
    END AS flag_reason
FROM with_plan_totals
WHERE default_wins_vs_plan > 0
ORDER BY provider_share_of_plan_defaults DESC

Provider_Email_Domain,Health_Plan_Issuer_Name,disputes_filed,default_wins_vs_plan,default_rate_vs_plan,plan_total_default_losses,provider_share_of_plan_defaults,flagged,flag_reason
usap.com,"KYOCERA DOCUMENT SOLUTIONS AMERICA, INC.",7,1,14.3,1,100.0,false,null
usap.com,"KTB Florida Sports Arena, LLC",1,1,100.0,1,100.0,true,"Default rate 100.0% vs plan, 100.0% of plan total defaults"
usap.com,"BAE SYSTEMS, INC.",74,10,13.5,10,100.0,false,null
summit-az.com,KYRENE SCHOOL DISTRICT,3,2,66.7,2,100.0,true,"Default rate 66.7% vs plan, 100.0% of plan total defaults"
usap.com,BACK OFFICE RISK,1,1,100.0,1,100.0,true,"Default rate 100.0% vs plan, 100.0% of plan total defaults"
usap.com,BALL CORPORATION,23,3,13.0,3,100.0,false,null
rightmedicalbilling.com,CIGNA HEALTH PLAN -- SELF FUNDED,1,1,100.0,1,100.0,true,"Default rate 100.0% vs plan, 100.0% of plan total defaults"
unitedionm.com,"CIGNA HEALTH PLAN, CIGNA",10,1,10.0,1,100.0,false,null
erevenuebilling.com,"Kaiser Foundation Health Plan ,Inc",1,1,100.0,1,100.0,true,"Default rate 100.0% vs plan, 100.0% of plan total defaults"
onet-systems.com,Kaiser - Multiplan,1,1,100.0,1,100.0,true,"Default rate 100.0% vs plan, 100.0% of plan total defaults"


In [0]:
%sql
-- ============================================================
-- Block 4 — Filing Cadence: Was There a Burst Pattern?
-- ============================================================
-- Quarter-over-quarter escalation using the 'quarter' column
-- from the combined table.

WITH quarterly AS (
    SELECT
        Provider_Email_Domain,
        quarter AS puf_quarter,
        COUNT(DISTINCT Dispute_Number) AS q_dispute_count,
        COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes'
             AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) AS q_default_wins,
        COUNT(DISTINCT CASE
            WHEN LOWER(TRIM(Type_of_Dispute)) = 'batched'
            THEN Dispute_Number END) AS q_batched_count
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain, quarter
),
q1 AS (
    SELECT * FROM quarterly WHERE LOWER(puf_quarter) LIKE '%q1%'
),
q2 AS (
    SELECT * FROM quarterly WHERE LOWER(puf_quarter) LIKE '%q2%'
)
SELECT
    COALESCE(q1.Provider_Email_Domain, q2.Provider_Email_Domain) AS Provider_Email_Domain,
    q1.q_dispute_count AS q1_disputes,
    q2.q_dispute_count AS q2_disputes,
    ROUND(100.0 * (q2.q_dispute_count - q1.q_dispute_count)
        / NULLIF(q1.q_dispute_count, 0), 1) AS dispute_change_pct,
    ROUND(100.0 * q1.q_default_wins / NULLIF(q1.q_dispute_count, 0), 1) AS q1_default_rate,
    ROUND(100.0 * q2.q_default_wins / NULLIF(q2.q_dispute_count, 0), 1) AS q2_default_rate,
    ROUND(100.0 * q2.q_default_wins / NULLIF(q2.q_dispute_count, 0), 1)
      - ROUND(100.0 * q1.q_default_wins / NULLIF(q1.q_dispute_count, 0), 1) AS default_rate_change_ppts,
    ROUND(100.0 * q1.q_batched_count / NULLIF(q1.q_dispute_count, 0), 1) AS q1_batched_share,
    ROUND(100.0 * q2.q_batched_count / NULLIF(q2.q_dispute_count, 0), 1) AS q2_batched_share,
    ROUND(100.0 * q2.q_batched_count / NULLIF(q2.q_dispute_count, 0), 1)
      - ROUND(100.0 * q1.q_batched_count / NULLIF(q1.q_dispute_count, 0), 1) AS batched_share_change_ppts,
    CASE
        WHEN 100.0 * (q2.q_dispute_count - q1.q_dispute_count) / NULLIF(q1.q_dispute_count, 0) > 25
          OR (
              ROUND(100.0 * q2.q_batched_count / NULLIF(q2.q_dispute_count, 0), 1)
              - ROUND(100.0 * q1.q_batched_count / NULLIF(q1.q_dispute_count, 0), 1)
             ) > 10
        THEN TRUE ELSE FALSE
    END AS flagged,
    CASE
        WHEN 100.0 * (q2.q_dispute_count - q1.q_dispute_count) / NULLIF(q1.q_dispute_count, 0) > 25
         AND (
              ROUND(100.0 * q2.q_batched_count / NULLIF(q2.q_dispute_count, 0), 1)
              - ROUND(100.0 * q1.q_batched_count / NULLIF(q1.q_dispute_count, 0), 1)
             ) > 10
        THEN CONCAT('Volume grew ', ROUND(100.0 * (q2.q_dispute_count - q1.q_dispute_count) / NULLIF(q1.q_dispute_count, 0), 1), '% AND batched share grew ',
                     ROUND(100.0 * q2.q_batched_count / NULLIF(q2.q_dispute_count, 0), 1) - ROUND(100.0 * q1.q_batched_count / NULLIF(q1.q_dispute_count, 0), 1), ' ppts')
        WHEN 100.0 * (q2.q_dispute_count - q1.q_dispute_count) / NULLIF(q1.q_dispute_count, 0) > 25
        THEN CONCAT('Volume grew ', ROUND(100.0 * (q2.q_dispute_count - q1.q_dispute_count) / NULLIF(q1.q_dispute_count, 0), 1), '% Q-over-Q')
        WHEN (
              ROUND(100.0 * q2.q_batched_count / NULLIF(q2.q_dispute_count, 0), 1)
              - ROUND(100.0 * q1.q_batched_count / NULLIF(q1.q_dispute_count, 0), 1)
             ) > 10
        THEN CONCAT('Batched share grew ',
                     ROUND(100.0 * q2.q_batched_count / NULLIF(q2.q_dispute_count, 0), 1) - ROUND(100.0 * q1.q_batched_count / NULLIF(q1.q_dispute_count, 0), 1), ' ppts Q-over-Q')
        ELSE NULL
    END AS flag_reason
FROM q1
FULL OUTER JOIN q2 ON q1.Provider_Email_Domain = q2.Provider_Email_Domain
WHERE q1.q_dispute_count IS NOT NULL
  AND q2.q_dispute_count IS NOT NULL
ORDER BY dispute_change_pct DESC NULLS LAST

Provider_Email_Domain,q1_disputes,q2_disputes,dispute_change_pct,q1_default_rate,q2_default_rate,default_rate_change_ppts,q1_batched_share,q2_batched_share,batched_share_change_ppts,flagged,flag_reason
chs.net,1,113,11200.0,0.0,97.3,97.3,0.0,0.0,0.0,true,Volume grew 11200.0% Q-over-Q
swiftstar.com,4,227,5575.0,0.0,51.1,51.1,0.0,0.0,0.0,true,Volume grew 5575.0% Q-over-Q
clearchoiceer.com,26,505,1842.3,15.4,46.9,31.5,0.0,0.0,0.0,true,Volume grew 1842.3% Q-over-Q
precisionmedarb.com,1,19,1800.0,0.0,5.3,5.3,100.0,78.9,-21.1,true,Volume grew 1800.0% Q-over-Q
gryphonhc.com,186,3264,1654.8,16.1,53.6,37.5,28.0,14.6,-13.4,true,Volume grew 1654.8% Q-over-Q
khcfirm.com,6,101,1583.3,0.0,6.9,6.9,0.0,8.9,8.9,true,Volume grew 1583.3% Q-over-Q
smh.com,2,31,1450.0,50.0,3.2,-46.8,50.0,35.5,-14.5,true,Volume grew 1450.0% Q-over-Q
aspirion.com,4,62,1450.0,0.0,0.0,0.0,0.0,0.0,0.0,true,Volume grew 1450.0% Q-over-Q
spinepainsolutions.com,11,150,1263.6,45.5,48.0,2.5,9.1,38.7,29.6,true,Volume grew 1263.6% AND batched share grew 29.6 ppts
mmbcontact.com,13,176,1253.8,30.8,12.5,-18.3,0.0,4.0,4.0,true,Volume grew 1253.8% Q-over-Q


In [0]:
%sql
-- ============================================================
-- Block 5 — Batched Dispute as a Volume Amplifier
-- ============================================================
-- Compare Single vs Batched: DLI density, default rate, merit
-- win rate. Flag providers whose batched filings produce
-- meaningfully more defaults than their single filings.

WITH dispute_type_stats AS (
    SELECT
        Provider_Email_Domain,
        LOWER(TRIM(Type_of_Dispute)) AS dispute_type,
        COUNT(DISTINCT Dispute_Number) AS dispute_count,
        COUNT(DLI_Number) AS dli_count,
        COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes'
             AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) AS default_wins,
        COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%'
             AND Default_Decision != 'Yes'
            THEN Dispute_Number END) AS merit_wins
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain, LOWER(TRIM(Type_of_Dispute))
),
single AS (
    SELECT
        Provider_Email_Domain,
        dispute_count AS single_disputes,
        ROUND(CAST(dli_count AS DOUBLE) / NULLIF(dispute_count, 0), 1) AS single_avg_dli,
        ROUND(100.0 * default_wins / NULLIF(dispute_count, 0), 1) AS single_default_rate,
        ROUND(100.0 * merit_wins / NULLIF(dispute_count, 0), 1) AS single_merit_win_rate
    FROM dispute_type_stats WHERE dispute_type = 'single'
),
batched AS (
    SELECT
        Provider_Email_Domain,
        dispute_count AS batched_disputes,
        ROUND(CAST(dli_count AS DOUBLE) / NULLIF(dispute_count, 0), 1) AS batched_avg_dli,
        ROUND(100.0 * default_wins / NULLIF(dispute_count, 0), 1) AS batched_default_rate,
        ROUND(100.0 * merit_wins / NULLIF(dispute_count, 0), 1) AS batched_merit_win_rate
    FROM dispute_type_stats WHERE dispute_type = 'batched'
)
SELECT
    COALESCE(s.Provider_Email_Domain, b.Provider_Email_Domain) AS Provider_Email_Domain,
    s.single_disputes,
    s.single_avg_dli,
    s.single_default_rate,
    s.single_merit_win_rate,
    b.batched_disputes,
    b.batched_avg_dli,
    b.batched_default_rate,
    b.batched_merit_win_rate,
    ROUND(b.batched_default_rate - s.single_default_rate, 1) AS batched_default_rate_premium,
    ROUND(b.batched_merit_win_rate - s.single_merit_win_rate, 1) AS batched_merit_win_rate_premium,
    CASE
        WHEN (b.batched_default_rate - s.single_default_rate) > 10
        THEN TRUE ELSE FALSE
    END AS flagged,
    CASE
        WHEN (b.batched_default_rate - s.single_default_rate) > 10
        THEN CONCAT('Batched default rate ', b.batched_default_rate, '% vs single ', s.single_default_rate, '% (+', ROUND(b.batched_default_rate - s.single_default_rate, 1), ' ppts premium)')
        ELSE NULL
    END AS flag_reason
FROM single s
FULL OUTER JOIN batched b ON s.Provider_Email_Domain = b.Provider_Email_Domain
WHERE s.single_disputes IS NOT NULL
  AND b.batched_disputes IS NOT NULL
ORDER BY batched_default_rate_premium DESC NULLS LAST

Provider_Email_Domain,single_disputes,single_avg_dli,single_default_rate,single_merit_win_rate,batched_disputes,batched_avg_dli,batched_default_rate,batched_merit_win_rate,batched_default_rate_premium,batched_merit_win_rate_premium,flagged,flag_reason
sonoranrm.comc,1,1.0,0.0,0.0,1,2.0,100.0,0.0,100.0,0.0,true,Batched default rate 100.0% vs single 0.0% (+100.0 ppts premium)
gottleibandgreenspan.com,10,1.0,10.0,20.0,1,6.0,100.0,0.0,90.0,-20.0,true,Batched default rate 100.0% vs single 10.0% (+90.0 ppts premium)
bergenanesthesiagroup.com,476,1.0,14.1,63.9,3,2.7,100.0,0.0,85.9,-63.9,true,Batched default rate 100.0% vs single 14.1% (+85.9 ppts premium)
macservicesllc.com,1483,1.0,18.3,66.6,1,4.0,100.0,0.0,81.7,-66.6,true,Batched default rate 100.0% vs single 18.3% (+81.7 ppts premium)
spinecarelongisland.com,8,1.0,25.0,62.5,1,2.0,100.0,0.0,75.0,-62.5,true,Batched default rate 100.0% vs single 25.0% (+75.0 ppts premium)
mountsinai.org,1,1.0,0.0,0.0,14,5.2,71.4,14.3,71.4,14.3,true,Batched default rate 71.4% vs single 0.0% (+71.4 ppts premium)
bracewell.com,27,1.3,37.0,44.4,2,7.0,100.0,0.0,63.0,-44.4,true,Batched default rate 100.0% vs single 37.0% (+63.0 ppts premium)
imail.org,47,1.0,40.4,48.9,3,8.0,100.0,0.0,59.6,-48.9,true,Batched default rate 100.0% vs single 40.4% (+59.6 ppts premium)
curamedgroup.com,21,1.0,47.6,33.3,1,2.0,100.0,0.0,52.4,-33.3,true,Batched default rate 100.0% vs single 47.6% (+52.4 ppts premium)
anthem.com,4,1.0,0.0,75.0,2,70.0,50.0,50.0,50.0,-25.0,true,Batched default rate 50.0% vs single 0.0% (+50.0 ppts premium)


In [0]:
%sql
-- ============================================================
-- Block 6 — Plan Victimization Ranking
-- ============================================================
-- Which health plans are bearing the highest burden from
-- high-volume filer defaults? Is the burden concentrated?
-- Includes HHI on default wins by provider and top-3 share.

WITH plan_disputes AS (
    SELECT
        Health_Plan_Issuer_Name,
        Dispute_Number,
        Default_Decision,
        Payment_Determination_Outcome,
        Provider_Email_Domain
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    GROUP BY Health_Plan_Issuer_Name, Dispute_Number, Default_Decision,
             Payment_Determination_Outcome, Provider_Email_Domain
),
plan_totals AS (
    SELECT
        Health_Plan_Issuer_Name,
        COUNT(DISTINCT Dispute_Number) AS total_disputes_against_plan,
        COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes'
            THEN Dispute_Number END) AS total_default_losses
    FROM plan_disputes
    GROUP BY Health_Plan_Issuer_Name
),
provider_defaults_at_plan AS (
    SELECT
        Health_Plan_Issuer_Name,
        Provider_Email_Domain,
        COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes'
             AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) AS provider_default_wins
    FROM plan_disputes
    WHERE Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Health_Plan_Issuer_Name, Provider_Email_Domain
),
with_shares AS (
    SELECT
        pdp.*,
        pt.total_default_losses,
        ROUND(100.0 * pdp.provider_default_wins / NULLIF(pt.total_default_losses, 0), 2) AS provider_default_share,
        ROW_NUMBER() OVER (
            PARTITION BY pdp.Health_Plan_Issuer_Name
            ORDER BY pdp.provider_default_wins DESC
        ) AS rank_at_plan
    FROM provider_defaults_at_plan pdp
    JOIN plan_totals pt ON pdp.Health_Plan_Issuer_Name = pt.Health_Plan_Issuer_Name
    WHERE pdp.provider_default_wins > 0
),
top3_share AS (
    SELECT
        Health_Plan_Issuer_Name,
        SUM(provider_default_share) AS top_3_provider_default_share
    FROM with_shares
    WHERE rank_at_plan <= 3
    GROUP BY Health_Plan_Issuer_Name
),
hhi_calc AS (
    SELECT
        Health_Plan_Issuer_Name,
        ROUND(SUM(POWER(provider_default_share / 100.0, 2)), 4) AS herfindahl_default_concentration
    FROM with_shares
    GROUP BY Health_Plan_Issuer_Name
)
SELECT
    pt.Health_Plan_Issuer_Name,
    pt.total_disputes_against_plan,
    pt.total_default_losses,
    ROUND(100.0 * pt.total_default_losses / NULLIF(pt.total_disputes_against_plan, 0), 1) AS plan_default_loss_rate,
    t3.top_3_provider_default_share,
    h.herfindahl_default_concentration,
    CASE
        WHEN 100.0 * pt.total_default_losses / NULLIF(pt.total_disputes_against_plan, 0) > 30
         AND t3.top_3_provider_default_share > 50
        THEN TRUE ELSE FALSE
    END AS flagged,
    CASE
        WHEN 100.0 * pt.total_default_losses / NULLIF(pt.total_disputes_against_plan, 0) > 30
         AND t3.top_3_provider_default_share > 50
        THEN CONCAT('Default loss rate ', ROUND(100.0 * pt.total_default_losses / NULLIF(pt.total_disputes_against_plan, 0), 1),
                     '% with top 3 providers causing ', t3.top_3_provider_default_share, '% of defaults. HHI: ', h.herfindahl_default_concentration)
        ELSE NULL
    END AS flag_reason
FROM plan_totals pt
LEFT JOIN top3_share t3 ON pt.Health_Plan_Issuer_Name = t3.Health_Plan_Issuer_Name
LEFT JOIN hhi_calc h ON pt.Health_Plan_Issuer_Name = h.Health_Plan_Issuer_Name
WHERE pt.total_disputes_against_plan > 100
ORDER BY plan_default_loss_rate DESC

Health_Plan_Issuer_Name,total_disputes_against_plan,total_default_losses,plan_default_loss_rate,top_3_provider_default_share,herfindahl_default_concentration,flagged,flag_reason
COMMUNITY HEALTH GROUP,103,103,100.0,100.00,1.0,true,Default loss rate 100.0% with top 3 providers causing 100.00% of defaults. HHI: 1.0
FRIDAY HEALTH PLANS,162,162,100.0,99.38,0.9876,true,Default loss rate 100.0% with top 3 providers causing 99.38% of defaults. HHI: 0.9876
PROMINENCE HEALTHFIRST,259,259,100.0,100.00,1.0,true,Default loss rate 100.0% with top 3 providers causing 100.00% of defaults. HHI: 1.0
SENTARA HEALTH PLANS,112,111,99.1,100.00,1.0,true,Default loss rate 99.1% with top 3 providers causing 100.00% of defaults. HHI: 1.0
CHRISTUS,154,152,98.7,98.68,0.7721,true,Default loss rate 98.7% with top 3 providers causing 98.68% of defaults. HHI: 0.7721
HMO PARTNERS DBA HEALTH ADVANTAGE,487,480,98.6,100.00,0.9958,true,Default loss rate 98.6% with top 3 providers causing 100.00% of defaults. HHI: 0.9958
PRESBYTERIAN HEALTH,181,178,98.3,100.00,1.0,true,Default loss rate 98.3% with top 3 providers causing 100.00% of defaults. HHI: 1.0
COMMUNITY CARE,361,355,98.3,99.99,0.9831,true,Default loss rate 98.3% with top 3 providers causing 99.99% of defaults. HHI: 0.9831
BCBS of PA,508,497,97.8,100.00,1.0,true,Default loss rate 97.8% with top 3 providers causing 100.00% of defaults. HHI: 1.0
Florida Blue,15367,13778,89.7,95.97,0.4267,true,Default loss rate 89.7% with top 3 providers causing 95.97% of defaults. HHI: 0.4267


In [0]:
%sql
-- ============================================================
-- Block 7 — The Dual-Track Operator List (THRESHOLDS FIXED)
-- ============================================================
-- Providers simultaneously running the merit-inflation track
-- (high volume, inflated awards) AND the predictable-default
-- track (high volume generating automatic wins).
-- Combines Block 1, Block 2, Block 3 logic.
--
-- CHANGES FROM ORIGINAL:
--   merit_inflation_flag: was merit_win_rate > 85 AND median_award > 400
--     Now: merit_win_rate > 60 AND median_award > 30 AND NOT isnan()
--     Rationale: P99 of actual median awards is ~126%. The >400%
--     threshold was impossible to trigger. 30% QPA is ~P75 of
--     the valid award distribution — a meaningful outlier cutoff.
--   Added isnan() filter to prevent NaN false positives.
--   volume_default_flag: unchanged (>20% default rate on >5K disputes).

WITH base_stats AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%'
             AND Default_Decision != 'Yes'
            THEN Dispute_Number END) AS merit_wins,
        COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes'
             AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) AS default_wins
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
),
extraction AS (
    SELECT
        Provider_Email_Domain,
        ROUND(PERCENTILE(
            TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE), 0.5
        ), 2) AS median_award_pct_qpa,
        ROUND(PERCENTILE(
            TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE), 0.5
        ) * COUNT(DLI_Number) / 100.0, 0) AS extraction_index
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Payment_Determination_Outcome ILIKE '%provider%'
      AND Default_Decision != 'Yes'
      AND Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
      AND TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE) IS NOT NULL
    GROUP BY Provider_Email_Domain
),
primary_idre AS (
    SELECT
        Provider_Email_Domain,
        idre_name AS primary_idre,
        ROW_NUMBER() OVER (
            PARTITION BY Provider_Email_Domain
            ORDER BY COUNT(*) DESC
        ) AS idre_rank
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Payment_Determination_Outcome ILIKE '%provider%'
      AND Default_Decision != 'Yes'
      AND Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
      AND idre_name IS NOT NULL
    GROUP BY Provider_Email_Domain, idre_name
),
combined AS (
    SELECT
        bs.Provider_Email_Domain,
        bs.total_disputes,
        bs.merit_wins,
        bs.default_wins,
        bs.merit_wins + bs.default_wins AS total_wins,
        ROUND(100.0 * bs.merit_wins / NULLIF(bs.total_disputes, 0), 1) AS merit_win_rate,
        ROUND(100.0 * bs.default_wins / NULLIF(bs.total_disputes, 0), 1) AS default_win_rate,
        ROUND(100.0 * (bs.merit_wins + bs.default_wins) / NULLIF(bs.total_disputes, 0), 1) AS combined_win_rate,
        ROUND(100.0 * bs.default_wins / NULLIF(bs.merit_wins + bs.default_wins, 0), 1) AS default_as_pct_of_total_wins,
        ex.median_award_pct_qpa,
        ex.extraction_index,
        pi.primary_idre,
        -- FIXED: relaxed from merit>85 AND award>400 to merit>60 AND award>30
        -- with NaN protection
        CASE
            WHEN 100.0 * bs.merit_wins / NULLIF(bs.total_disputes, 0) > 60
             AND ex.median_award_pct_qpa IS NOT NULL
             AND NOT isnan(ex.median_award_pct_qpa)
             AND ex.median_award_pct_qpa > 30
            THEN 1 ELSE 0
        END AS merit_inflation_flag,
        CASE
            WHEN 100.0 * bs.default_wins / NULLIF(bs.total_disputes, 0) > 20
             AND bs.total_disputes > 5000
            THEN 1 ELSE 0
        END AS volume_default_flag
    FROM base_stats bs
    LEFT JOIN extraction ex ON bs.Provider_Email_Domain = ex.Provider_Email_Domain
    LEFT JOIN primary_idre pi ON bs.Provider_Email_Domain = pi.Provider_Email_Domain
                              AND pi.idre_rank = 1
)
SELECT
    Provider_Email_Domain,
    total_disputes,
    merit_wins,
    default_wins,
    combined_win_rate,
    merit_win_rate,
    default_win_rate,
    default_as_pct_of_total_wins,
    median_award_pct_qpa,
    extraction_index,
    primary_idre,
    merit_inflation_flag,
    volume_default_flag,
    CASE
        WHEN merit_inflation_flag = 1 AND volume_default_flag = 1 THEN 'Dual-Track'
        WHEN merit_inflation_flag = 1 THEN 'Merit Inflation Only'
        WHEN volume_default_flag = 1 THEN 'Default Gaming Only'
        ELSE 'Clean'
    END AS track_type,
    CASE
        WHEN merit_inflation_flag = 1 AND volume_default_flag = 1 THEN TRUE
        WHEN merit_inflation_flag = 1 OR volume_default_flag = 1 THEN TRUE
        ELSE FALSE
    END AS flagged,
    CASE
        WHEN merit_inflation_flag = 1 AND volume_default_flag = 1
        THEN CONCAT('DUAL-TRACK: Merit rate ', merit_win_rate, '%, median award ', median_award_pct_qpa, '% QPA, default rate ', default_win_rate, '% on ', total_disputes, ' disputes. Primary IDRE: ', COALESCE(primary_idre, 'N/A'))
        WHEN merit_inflation_flag = 1
        THEN CONCAT('MERIT INFLATION: Merit rate ', merit_win_rate, '%, median award ', median_award_pct_qpa, '% QPA')
        WHEN volume_default_flag = 1
        THEN CONCAT('DEFAULT GAMING: Default rate ', default_win_rate, '% on ', total_disputes, ' disputes')
        ELSE NULL
    END AS flag_reason
FROM combined
WHERE merit_inflation_flag = 1 OR volume_default_flag = 1
ORDER BY
    CASE
        WHEN merit_inflation_flag = 1 AND volume_default_flag = 1 THEN 1
        WHEN merit_inflation_flag = 1 THEN 2
        WHEN volume_default_flag = 1 THEN 3
        ELSE 4
    END,
    total_disputes DESC,
    extraction_index DESC

Provider_Email_Domain,total_disputes,merit_wins,default_wins,combined_win_rate,merit_win_rate,default_win_rate,default_as_pct_of_total_wins,median_award_pct_qpa,extraction_index,primary_idre,merit_inflation_flag,volume_default_flag,track_type,flagged,flag_reason
mdcapitaladvisors.com,20180,15558,2747,90.7,77.1,13.6,15.0,44.68,7580.0,"Maximus Federal Services, Inc.",1,0,Merit Inflation Only,true,"MERIT INFLATION: Merit rate 77.1%, median award 44.68% QPA"
gottliebandgreenspan.com,15017,10465,2825,88.5,69.7,18.8,21.3,41.99,3209.0,"EdiPhy Advisors, LLC",1,0,Merit Inflation Only,true,"MERIT INFLATION: Merit rate 69.7%, median award 41.99% QPA"
halkovichlaw.com,5811,4113,1036,88.6,70.8,17.8,20.1,39.39,2006.0,"C2C Innovative Solutions, Inc.",1,0,Merit Inflation Only,true,"MERIT INFLATION: Merit rate 70.8%, median award 39.39% QPA"
mlanesthesia.com,834,643,89,87.8,77.1,10.7,12.2,34.29,220.0,"C2C Innovative Solutions, Inc.",1,0,Merit Inflation Only,true,"MERIT INFLATION: Merit rate 77.1%, median award 34.29% QPA"
nsabilling.com,537,372,127,92.9,69.3,23.6,25.5,33.82,136.0,"C2C Innovative Solutions, Inc.",1,0,Merit Inflation Only,true,"MERIT INFLATION: Merit rate 69.3%, median award 33.82% QPA"
medbill-assoc.com,421,278,84,86.0,66.0,20.0,23.2,39.15,135.0,"Keystone Peer Review Organization, Inc.",1,0,Merit Inflation Only,true,"MERIT INFLATION: Merit rate 66.0%, median award 39.15% QPA"
recoveryspecialist.org,342,262,49,90.9,76.6,14.3,15.8,33.77,80.0,Island Peer Review Organization (DBA: IPRO),1,0,Merit Inflation Only,true,"MERIT INFLATION: Merit rate 76.6%, median award 33.77% QPA"
solestra.net,232,170,36,88.8,73.3,15.5,17.5,37.69,69.0,"C2C Innovative Solutions, Inc.",1,0,Merit Inflation Only,true,"MERIT INFLATION: Merit rate 73.3%, median award 37.69% QPA"
atlanticneurocare.com,190,136,24,84.2,71.6,12.6,15.0,51.18,91.0,"Capitol Bridge, LLC",1,0,Merit Inflation Only,true,"MERIT INFLATION: Merit rate 71.6%, median award 51.18% QPA"
mmbcontact.com,189,135,26,85.2,71.4,13.8,16.1,53.38,74.0,"C2C Innovative Solutions, Inc.",1,0,Merit Inflation Only,true,"MERIT INFLATION: Merit rate 71.4%, median award 53.38% QPA"


In [0]:
%sql
-- ============================================================
-- Block 8 — Quarter-over-Quarter Persistence Validation
-- ============================================================
-- Which providers appear on the high-volume/high-extraction list
-- in BOTH Q1 and Q2? Confirms persistent operational strategy.

WITH q1_stats AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS q1_disputes,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%'
             AND Default_Decision != 'Yes'
            THEN Dispute_Number END)
            / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS q1_merit_win_rate,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes'
             AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END)
            / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS q1_default_win_rate
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE LOWER(quarter) LIKE '%q1%'
      AND Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
),
q2_stats AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS q2_disputes,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%'
             AND Default_Decision != 'Yes'
            THEN Dispute_Number END)
            / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS q2_merit_win_rate,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes'
             AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END)
            / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS q2_default_win_rate
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE LOWER(quarter) LIKE '%q2%'
      AND Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
),
dual_track_list AS (
    SELECT DISTINCT Provider_Email_Domain
    FROM (
        SELECT
            Provider_Email_Domain,
            COUNT(DISTINCT Dispute_Number) AS total_disputes,
            ROUND(100.0 * COUNT(DISTINCT CASE
                WHEN Payment_Determination_Outcome ILIKE '%provider%'
                 AND Default_Decision != 'Yes'
                THEN Dispute_Number END)
                / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS merit_win_rate,
            ROUND(100.0 * COUNT(DISTINCT CASE
                WHEN Default_Decision = 'Yes'
                 AND Payment_Determination_Outcome ILIKE '%provider%'
                THEN Dispute_Number END)
                / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS default_win_rate
        FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
        WHERE Provider_Email_Domain IS NOT NULL
          AND TRIM(Provider_Email_Domain) != ''
        GROUP BY Provider_Email_Domain
    ) sub
    WHERE (merit_win_rate > 85 AND total_disputes > 1000)
       OR (default_win_rate > 20 AND total_disputes > 5000)
)
SELECT
    COALESCE(q1.Provider_Email_Domain, q2.Provider_Email_Domain) AS Provider_Email_Domain,
    CASE WHEN q1.q1_disputes IS NOT NULL AND q2.q2_disputes IS NOT NULL THEN TRUE ELSE FALSE END AS present_both_quarters,
    q1.q1_disputes,
    q2.q2_disputes,
    COALESCE(q1.q1_disputes, 0) + COALESCE(q2.q2_disputes, 0) AS total_disputes_combined,
    ROUND(100.0 * (COALESCE(q2.q2_disputes, 0) - COALESCE(q1.q1_disputes, 0))
        / NULLIF(q1.q1_disputes, 0), 1) AS q_over_q_growth_pct,
    q1.q1_merit_win_rate,
    q2.q2_merit_win_rate,
    q1.q1_default_win_rate,
    q2.q2_default_win_rate,
    CASE WHEN dtl.Provider_Email_Domain IS NOT NULL THEN TRUE ELSE FALSE END AS on_dual_track_list,
    CASE
        WHEN q1.q1_disputes IS NOT NULL
         AND q2.q2_disputes IS NOT NULL
         AND dtl.Provider_Email_Domain IS NOT NULL
        THEN 'Confirmed Persistent Dual-Track Operator'
        WHEN q1.q1_disputes IS NOT NULL
         AND q2.q2_disputes IS NOT NULL
        THEN 'Persistent Operator (both quarters)'
        ELSE 'Single Quarter Only'
    END AS persistence_status,
    CASE
        WHEN q1.q1_disputes IS NOT NULL
         AND q2.q2_disputes IS NOT NULL
         AND (COALESCE(q1.q1_disputes, 0) + COALESCE(q2.q2_disputes, 0)) > 5000
        THEN TRUE ELSE FALSE
    END AS flagged,
    CASE
        WHEN q1.q1_disputes IS NOT NULL
         AND q2.q2_disputes IS NOT NULL
         AND dtl.Provider_Email_Domain IS NOT NULL
        THEN CONCAT('PERSISTENT DUAL-TRACK: Q1 ', q1.q1_disputes, ' disputes, Q2 ', q2.q2_disputes, ' disputes. Merit: Q1 ', q1.q1_merit_win_rate, '% / Q2 ', q2.q2_merit_win_rate, '%. Default: Q1 ', q1.q1_default_win_rate, '% / Q2 ', q2.q2_default_win_rate, '%')
        WHEN q1.q1_disputes IS NOT NULL
         AND q2.q2_disputes IS NOT NULL
        THEN CONCAT('Present both quarters: Q1 ', q1.q1_disputes, ', Q2 ', q2.q2_disputes, ' (', ROUND(100.0 * (COALESCE(q2.q2_disputes, 0) - COALESCE(q1.q1_disputes, 0)) / NULLIF(q1.q1_disputes, 0), 1), '% growth)')
        ELSE NULL
    END AS flag_reason
FROM q1_stats q1
FULL OUTER JOIN q2_stats q2 ON q1.Provider_Email_Domain = q2.Provider_Email_Domain
LEFT JOIN dual_track_list dtl ON COALESCE(q1.Provider_Email_Domain, q2.Provider_Email_Domain) = dtl.Provider_Email_Domain
WHERE (q1.q1_disputes IS NOT NULL AND q2.q2_disputes IS NOT NULL)
  AND (COALESCE(q1.q1_disputes, 0) + COALESCE(q2.q2_disputes, 0)) > 5000
ORDER BY
    CASE WHEN dtl.Provider_Email_Domain IS NOT NULL THEN 1 ELSE 2 END,
    total_disputes_combined DESC

Provider_Email_Domain,present_both_quarters,q1_disputes,q2_disputes,total_disputes_combined,q_over_q_growth_pct,q1_merit_win_rate,q2_merit_win_rate,q1_default_win_rate,q2_default_win_rate,on_dual_track_list,persistence_status,flagged,flag_reason
halomd.com,true,75060,105646,180706,40.7,65.7,59.1,22.1,27.9,true,Confirmed Persistent Dual-Track Operator,true,"PERSISTENT DUAL-TRACK: Q1 75060 disputes, Q2 105646 disputes. Merit: Q1 65.7% / Q2 59.1%. Default: Q1 22.1% / Q2 27.9%"
teamhealth.com,true,73657,95142,168799,29.2,84.6,88.9,9.1,6.6,true,Confirmed Persistent Dual-Track Operator,true,"PERSISTENT DUAL-TRACK: Q1 73657 disputes, Q2 95142 disputes. Merit: Q1 84.6% / Q2 88.9%. Default: Q1 9.1% / Q2 6.6%"
agshealth.com,true,23085,27337,50422,18.4,61.4,68.3,25.7,17.5,true,Confirmed Persistent Dual-Track Operator,true,"PERSISTENT DUAL-TRACK: Q1 23085 disputes, Q2 27337 disputes. Merit: Q1 61.4% / Q2 68.3%. Default: Q1 25.7% / Q2 17.5%"
scp-health.com,true,22804,24419,47223,7.1,65.2,69.2,27.1,18.1,true,Confirmed Persistent Dual-Track Operator,true,"PERSISTENT DUAL-TRACK: Q1 22804 disputes, Q2 24419 disputes. Merit: Q1 65.2% / Q2 69.2%. Default: Q1 27.1% / Q2 18.1%"
fam-llc.com,true,9727,28318,38045,191.1,32.4,41.4,46.1,42.1,true,Confirmed Persistent Dual-Track Operator,true,"PERSISTENT DUAL-TRACK: Q1 9727 disputes, Q2 28318 disputes. Merit: Q1 32.4% / Q2 41.4%. Default: Q1 46.1% / Q2 42.1%"
orthomedstaffing.com,true,17648,13277,30925,-24.8,66.2,61.5,20.3,22.9,true,Confirmed Persistent Dual-Track Operator,true,"PERSISTENT DUAL-TRACK: Q1 17648 disputes, Q2 13277 disputes. Merit: Q1 66.2% / Q2 61.5%. Default: Q1 20.3% / Q2 22.9%"
qmacsmso.com,true,10984,19597,30581,78.4,36.5,43.6,39.3,39.6,true,Confirmed Persistent Dual-Track Operator,true,"PERSISTENT DUAL-TRACK: Q1 10984 disputes, Q2 19597 disputes. Merit: Q1 36.5% / Q2 43.6%. Default: Q1 39.3% / Q2 39.6%"
radpmg.com,true,12169,13786,25955,13.3,78.7,95.1,15.7,2.8,true,Confirmed Persistent Dual-Track Operator,true,"PERSISTENT DUAL-TRACK: Q1 12169 disputes, Q2 13786 disputes. Merit: Q1 78.7% / Q2 95.1%. Default: Q1 15.7% / Q2 2.8%"
usap.com,true,7184,16662,23846,131.9,49.8,74.6,41.2,12.1,true,Confirmed Persistent Dual-Track Operator,true,"PERSISTENT DUAL-TRACK: Q1 7184 disputes, Q2 16662 disputes. Merit: Q1 49.8% / Q2 74.6%. Default: Q1 41.2% / Q2 12.1%"
specialtycare.net,true,8894,9012,17906,1.3,58.7,72.9,31.2,13.9,true,Confirmed Persistent Dual-Track Operator,true,"PERSISTENT DUAL-TRACK: Q1 8894 disputes, Q2 9012 disputes. Merit: Q1 58.7% / Q2 72.9%. Default: Q1 31.2% / Q2 13.9%"


In [0]:
%sql
-- ============================================================
-- Block 9 — Final Cross-Reference Audit Ranking (UPDATED)
-- ============================================================
-- Incorporates all threshold fixes from analysis:
--   Block 2: med_award > 30 (was >500, unreachable). NaN filter added.
--   Block 3: Minimum 25 disputes per provider-plan pair.
--   Block 5: Minimum 50 batched disputes to filter noise.
--   Block 6: Counts only provider-won defaults.
--   Block 7: Now matches updated Block 7 logic exactly:
--            merit_inflation = merit>60% AND median_award>30% (NaN-safe)
--            volume_default  = default>20% AND disputes>5K
--            dual_track       = BOTH flags triggered
--   New: Volume-weighted risk_score = flags * log(disputes) * (1 + default_rate/100)
--   Tier 1: 5+ flags | Tier 2: 3-4 | Monitoring: 1-2

WITH block1_flags AS (
    SELECT DISTINCT Provider_Email_Domain, 1 AS b1
    FROM (
        SELECT Provider_Email_Domain, COUNT(DISTINCT Dispute_Number) AS td
        FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
        WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
        GROUP BY Provider_Email_Domain
    ) t
    CROSS JOIN (
        SELECT COUNT(DISTINCT Dispute_Number) AS nat
        FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    ) n
    WHERE 100.0 * t.td / n.nat > 1.0 OR t.td > 5000
),
-- Block 2 FIXED: threshold lowered to P90, NaN filtered
block2_flags AS (
    SELECT Provider_Email_Domain, 1 AS b2
    FROM (
        SELECT Provider_Email_Domain,
               PERCENTILE(TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE), 0.5) AS med
        FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
        WHERE Payment_Determination_Outcome ILIKE '%provider%'
          AND Default_Decision != 'Yes'
          AND Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
          AND TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE) IS NOT NULL
        GROUP BY Provider_Email_Domain
    ) t
    WHERE med IS NOT NULL AND NOT isnan(med) AND med > 30
),
-- Block 3 FIXED: minimum 25 disputes per pair
block3_flags AS (
    SELECT DISTINCT Provider_Email_Domain, 1 AS b3
    FROM (
        SELECT Provider_Email_Domain, Health_Plan_Issuer_Name,
               COUNT(DISTINCT Dispute_Number) AS df,
               COUNT(DISTINCT CASE WHEN Default_Decision = 'Yes'
                   AND Payment_Determination_Outcome ILIKE '%provider%'
                   THEN Dispute_Number END) AS dw
        FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
        WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
        GROUP BY Provider_Email_Domain, Health_Plan_Issuer_Name
    ) pp
    WHERE df >= 25 AND 100.0 * dw / NULLIF(df, 0) > 40 AND dw > 10
),
block4_flags AS (
    SELECT DISTINCT Provider_Email_Domain, 1 AS b4
    FROM (
        SELECT Provider_Email_Domain,
            SUM(CASE WHEN LOWER(quarter) LIKE '%q1%' THEN 1 ELSE 0 END) AS q1,
            SUM(CASE WHEN LOWER(quarter) LIKE '%q2%' THEN 1 ELSE 0 END) AS q2
        FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
        WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
        GROUP BY Provider_Email_Domain
    ) t
    WHERE 100.0 * (q2 - q1) / NULLIF(q1, 0) > 25
),
-- Block 5 FIXED: minimum 50 batched disputes
block5_flags AS (
    SELECT DISTINCT Provider_Email_Domain, 1 AS b5
    FROM (
        SELECT Provider_Email_Domain, LOWER(TRIM(Type_of_Dispute)) AS dtype,
            COUNT(DISTINCT Dispute_Number) AS dcnt,
            COUNT(DISTINCT CASE WHEN Default_Decision = 'Yes'
                AND Payment_Determination_Outcome ILIKE '%provider%'
                THEN Dispute_Number END) AS dw
        FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
        WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
        GROUP BY Provider_Email_Domain, LOWER(TRIM(Type_of_Dispute))
    ) t
    PIVOT (
        MAX(dcnt) AS cnt,
        MAX(100.0 * dw / NULLIF(dcnt, 0)) AS rate
        FOR dtype IN ('single', 'batched')
    )
    WHERE batched_cnt >= 50 AND batched_rate - single_rate > 10
),
-- Block 6 FIXED: counts only provider-won defaults
block6_flags AS (
    SELECT DISTINCT Provider_Email_Domain, 1 AS b6
    FROM (
        SELECT Provider_Email_Domain, Health_Plan_Issuer_Name,
            COUNT(DISTINCT CASE WHEN Default_Decision = 'Yes'
                AND Payment_Determination_Outcome ILIKE '%provider%'
                THEN Dispute_Number END) AS dw
        FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
        WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
        GROUP BY Provider_Email_Domain, Health_Plan_Issuer_Name
    ) t
    WHERE dw > 50
),
-- Block 7 FIXED: matches updated Block 7 dual-track logic exactly
-- merit_inflation = merit>60% AND median_award>30% QPA (NaN-safe)
-- volume_default  = default>20% AND total_disputes>5K
-- A provider is flagged if EITHER condition is met
block7_flags AS (
    SELECT t.Provider_Email_Domain, 1 AS b7
    FROM (
        SELECT Provider_Email_Domain,
            COUNT(DISTINCT Dispute_Number) AS td,
            100.0 * COUNT(DISTINCT CASE
                WHEN Payment_Determination_Outcome ILIKE '%provider%'
                 AND Default_Decision != 'Yes'
                THEN Dispute_Number END)
                / NULLIF(COUNT(DISTINCT Dispute_Number), 0) AS mwr,
            100.0 * COUNT(DISTINCT CASE
                WHEN Default_Decision = 'Yes'
                 AND Payment_Determination_Outcome ILIKE '%provider%'
                THEN Dispute_Number END)
                / NULLIF(COUNT(DISTINCT Dispute_Number), 0) AS dwr
        FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
        WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
        GROUP BY Provider_Email_Domain
    ) t
    LEFT JOIN (
        SELECT Provider_Email_Domain,
               PERCENTILE(TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE), 0.5) AS med
        FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
        WHERE Payment_Determination_Outcome ILIKE '%provider%'
          AND Default_Decision != 'Yes'
          AND Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
          AND TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE) IS NOT NULL
        GROUP BY Provider_Email_Domain
    ) ex ON t.Provider_Email_Domain = ex.Provider_Email_Domain
    WHERE
        -- Merit inflation track: high merit rate + elevated awards
        (mwr > 60 AND ex.med IS NOT NULL AND NOT isnan(ex.med) AND ex.med > 30)
        OR
        -- Volume default track: high default rate at scale
        (dwr > 20 AND td > 5000)
),
block8_flags AS (
    SELECT DISTINCT Provider_Email_Domain, 1 AS b8
    FROM (
        SELECT Provider_Email_Domain, quarter,
               COUNT(DISTINCT Dispute_Number) AS dcnt
        FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
        WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
        GROUP BY Provider_Email_Domain, quarter
    ) t
    GROUP BY Provider_Email_Domain
    HAVING COUNT(DISTINCT quarter) >= 2 AND SUM(dcnt) > 5000
),
-- Volume data for risk_score weighting
volumes AS (
    SELECT Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes'
             AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END)
            / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS default_rate
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
),
combined AS (
    SELECT v.Provider_Email_Domain, v.total_disputes, v.default_rate,
        COALESCE(b1.b1, 0) AS b1,
        COALESCE(b2.b2, 0) AS b2,
        COALESCE(b3.b3, 0) AS b3,
        COALESCE(b4.b4, 0) AS b4,
        COALESCE(b5.b5, 0) AS b5,
        COALESCE(b6.b6, 0) AS b6,
        COALESCE(b7.b7, 0) AS b7,
        COALESCE(b8.b8, 0) AS b8,
        (COALESCE(b1.b1,0) + COALESCE(b2.b2,0) + COALESCE(b3.b3,0)
       + COALESCE(b4.b4,0) + COALESCE(b5.b5,0) + COALESCE(b6.b6,0)
       + COALESCE(b7.b7,0) + COALESCE(b8.b8,0)) AS total_flags
    FROM volumes v
    LEFT JOIN block1_flags b1 ON v.Provider_Email_Domain = b1.Provider_Email_Domain
    LEFT JOIN block2_flags b2 ON v.Provider_Email_Domain = b2.Provider_Email_Domain
    LEFT JOIN block3_flags b3 ON v.Provider_Email_Domain = b3.Provider_Email_Domain
    LEFT JOIN block4_flags b4 ON v.Provider_Email_Domain = b4.Provider_Email_Domain
    LEFT JOIN block5_flags b5 ON v.Provider_Email_Domain = b5.Provider_Email_Domain
    LEFT JOIN block6_flags b6 ON v.Provider_Email_Domain = b6.Provider_Email_Domain
    LEFT JOIN block7_flags b7 ON v.Provider_Email_Domain = b7.Provider_Email_Domain
    LEFT JOIN block8_flags b8 ON v.Provider_Email_Domain = b8.Provider_Email_Domain
)
SELECT *,
    -- Volume-weighted risk score: flags * log(disputes) * (1 + default_rate/100)
    ROUND(total_flags * LOG10(GREATEST(total_disputes, 1)) * (1 + default_rate / 100.0), 1) AS risk_score,
    CASE
        WHEN total_flags >= 5 THEN 'Tier 1 - Audit Referral'
        WHEN total_flags BETWEEN 3 AND 4 THEN 'Tier 2 - Enhanced Review'
        WHEN total_flags BETWEEN 1 AND 2 THEN 'Monitoring'
        ELSE 'Clean'
    END AS audit_tier
FROM combined
WHERE total_flags >= 1
ORDER BY risk_score DESC, total_flags DESC

Provider_Email_Domain,total_disputes,default_rate,b1,b2,b3,b4,b5,b6,b7,b8,total_flags,risk_score,audit_tier
fam-llc.com,38045,43.1,1,0,1,1,1,1,1,1,7,45.9,Tier 1 - Audit Referral
halomd.com,180706,25.5,1,0,1,1,0,1,1,1,6,39.6,Tier 1 - Audit Referral
totalcare.us,14006,35.2,1,0,1,1,1,1,1,1,7,39.2,Tier 1 - Audit Referral
qmacsmso.com,30581,39.5,1,0,1,1,0,1,1,1,6,37.5,Tier 1 - Audit Referral
usap.com,23846,20.9,1,0,1,1,1,1,1,1,7,37.0,Tier 1 - Audit Referral
altushealthsystem.com,9379,50.8,1,0,1,1,0,1,1,1,6,35.9,Tier 1 - Audit Referral
gottliebandgreenspan.com,15017,18.8,1,1,1,1,0,1,1,1,7,34.7,Tier 1 - Audit Referral
mdcapitaladvisors.com,20180,13.6,1,1,1,1,0,1,1,1,7,34.2,Tier 1 - Audit Referral
primehealthcare.com,16470,32.8,1,0,1,1,0,1,1,1,6,33.6,Tier 1 - Audit Referral
aimbillingsolutions.com,6323,46.1,1,0,1,1,0,1,1,1,6,33.3,Tier 1 - Audit Referral


In [0]:
%sql
-- ============================================================
-- Threshold Optimization: Distribution Analysis & Sensitivity
-- ============================================================

-- 1. Provider-level metric distributions (find natural break points)
WITH provider_core AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%' AND Default_Decision != 'Yes'
            THEN Dispute_Number END) / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS merit_rate,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes' AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS default_rate
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
),
national AS (
    SELECT COUNT(DISTINCT Dispute_Number) AS national_total
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
),
-- 2. Valid award distribution (critical for fixing Block 2/7)
award_dist AS (
    SELECT
        Provider_Email_Domain,
        PERCENTILE(TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE), 0.5) AS med_award
    FROM idre.idre_silver.oon_emergency_nonemergency_fee_join
    WHERE Payment_Determination_Outcome ILIKE '%provider%'
      AND Default_Decision != 'Yes'
      AND Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
      AND TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE) IS NOT NULL
    GROUP BY Provider_Email_Domain
),
valid_awards AS (
    SELECT * FROM award_dist WHERE med_award IS NOT NULL AND NOT isnan(med_award)
)
SELECT
    '=== VOLUME DISTRIBUTION ===' AS section,
    PERCENTILE(CAST(total_disputes AS DOUBLE), 0.50) AS p50,
    PERCENTILE(CAST(total_disputes AS DOUBLE), 0.75) AS p75,
    PERCENTILE(CAST(total_disputes AS DOUBLE), 0.90) AS p90,
    PERCENTILE(CAST(total_disputes AS DOUBLE), 0.95) AS p95,
    PERCENTILE(CAST(total_disputes AS DOUBLE), 0.99) AS p99,
    NULL AS provider_count
FROM provider_core

UNION ALL

SELECT
    '=== DEFAULT RATE DISTRIBUTION ===' AS section,
    PERCENTILE(default_rate, 0.50), PERCENTILE(default_rate, 0.75),
    PERCENTILE(default_rate, 0.90), PERCENTILE(default_rate, 0.95),
    PERCENTILE(default_rate, 0.99), NULL
FROM provider_core
WHERE total_disputes > 1000

UNION ALL

SELECT
    '=== MERIT RATE DISTRIBUTION ===' AS section,
    PERCENTILE(merit_rate, 0.50), PERCENTILE(merit_rate, 0.75),
    PERCENTILE(merit_rate, 0.90), PERCENTILE(merit_rate, 0.95),
    PERCENTILE(merit_rate, 0.99), NULL
FROM provider_core
WHERE total_disputes > 1000

UNION ALL

SELECT
    '=== MEDIAN AWARD DISTRIBUTION (valid only) ===' AS section,
    PERCENTILE(med_award, 0.50), PERCENTILE(med_award, 0.75),
    PERCENTILE(med_award, 0.90), PERCENTILE(med_award, 0.95),
    PERCENTILE(med_award, 0.99),
    CAST((SELECT COUNT(*) FROM valid_awards) AS DOUBLE)
FROM valid_awards

UNION ALL

-- Dual-track sensitivity: how many meet BOTH conditions at various thresholds?
SELECT
    'DualTrack: merit>70 AND default>15 (>1K disputes)' AS section,
    NULL, NULL, NULL, NULL, NULL,
    CAST(COUNT(*) AS DOUBLE)
FROM provider_core WHERE merit_rate > 70 AND default_rate > 15 AND total_disputes > 1000

UNION ALL

SELECT
    'DualTrack: merit>60 AND default>20 (>1K disputes)' AS section,
    NULL, NULL, NULL, NULL, NULL,
    CAST(COUNT(*) AS DOUBLE)
FROM provider_core WHERE merit_rate > 60 AND default_rate > 20 AND total_disputes > 1000

UNION ALL

SELECT
    'DualTrack: merit>60 AND default>25 (>5K disputes)' AS section,
    NULL, NULL, NULL, NULL, NULL,
    CAST(COUNT(*) AS DOUBLE)
FROM provider_core WHERE merit_rate > 60 AND default_rate > 25 AND total_disputes > 5000

section,p50,p75,p90,p95,p99,provider_count
=== VOLUME DISTRIBUTION ===,10.0,166.0,2045.400000000001,7133.199999999998,38779.23999999985,null
=== DEFAULT RATE DISTRIBUTION ===,20.2,31.0,42.660000000000004,48.35999999999999,51.088,null
=== MERIT RATE DISTRIBUTION ===,62.5,69.9,80.94,82.22,87.144,null
=== MEDIAN AWARD DISTRIBUTION (valid only) ===,6.4925,21.965,44.14700000000006,64.99724999999997,125.63464999999971,342.0
DualTrack: merit>70 AND default>15 (>1K disputes),null,null,null,null,null,2.0
DualTrack: merit>60 AND default>20 (>1K disputes),null,null,null,null,null,13.0
DualTrack: merit>60 AND default>25 (>5K disputes),null,null,null,null,null,2.0


# Medicare Fee Schedule Enhancement — Signals A–E

**New Primary Table:** `idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency`

This section leverages Medicare fee schedule data (CY2025) joined at the `Service_Code` (HCPCS/CPT) level to anchor extraction analysis against an **externally validated benchmark** — Medicare reimbursement rates set by CMS itself.

### Key New Columns Available
| Column | Description |
|---|---|
| `qpa_as_pct_of_medicare` | QPA offered by plan as % of Medicare rate |
| `provider_offer_as_pct_of_medicare` | Provider's ask as % of Medicare rate |
| `flag_provider_over_3x_medicare` | Boolean: provider asking >3× Medicare |
| `flag_qpa_below_50pct_medicare` | Boolean: QPA depressed below 50% of Medicare |
| `medicare_non_facility_rate` | Actual Medicare $ rate per code |
| `total_non_facility_rvu` | Clinical complexity proxy (RVU) |
| `Service_Code` | CPT/HCPCS code per DLI |

### Why This Matters
- Medicare rates are **externally anchored** — set by CMS, not negotiated between parties
- Moves from relative (QPA%) to absolute ($ above Medicare) extraction estimates
- Enables service-code-level pattern detection (template billing)

In [0]:
%sql
-- ============================================================
-- VALIDATION — Medicare Fee Schedule Join Quality
-- ============================================================
-- Checks coverage and null rates in the new joined table vs original.
-- Critical: how many DLIs have valid Medicare rates attached?

WITH coverage AS (
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        COUNT(DISTINCT DLI_Number) AS total_dlis,
        
        -- Medicare rate coverage
        COUNT(CASE WHEN medicare_non_facility_rate IS NOT NULL THEN 1 END) AS rows_with_medicare_rate,
        COUNT(CASE WHEN medicare_facility_rate IS NOT NULL THEN 1 END) AS rows_with_facility_rate,
        COUNT(CASE WHEN Service_Code IS NOT NULL AND TRIM(Service_Code) != '' THEN 1 END) AS rows_with_service_code,
        
        -- Key derived columns
        COUNT(CASE WHEN qpa_as_pct_of_medicare IS NOT NULL THEN 1 END) AS rows_with_qpa_pct_medicare,
        COUNT(CASE WHEN provider_offer_as_pct_of_medicare IS NOT NULL THEN 1 END) AS rows_with_provider_offer_pct,
        
        -- Pre-computed flags
        COUNT(CASE WHEN flag_provider_over_3x_medicare = TRUE THEN 1 END) AS flagged_provider_over_3x,
        COUNT(CASE WHEN flag_qpa_below_50pct_medicare = TRUE THEN 1 END) AS flagged_qpa_below_50pct,
        
        -- RVU coverage
        COUNT(CASE WHEN total_non_facility_rvu IS NOT NULL THEN 1 END) AS rows_with_rvu
    FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
)
SELECT
    total_rows,
    total_disputes,
    total_dlis,
    rows_with_medicare_rate,
    ROUND(100.0 * rows_with_medicare_rate / total_rows, 1) AS pct_with_medicare_rate,
    rows_with_service_code,
    ROUND(100.0 * rows_with_service_code / total_rows, 1) AS pct_with_service_code,
    rows_with_qpa_pct_medicare,
    ROUND(100.0 * rows_with_qpa_pct_medicare / total_rows, 1) AS pct_with_qpa_pct_medicare,
    rows_with_provider_offer_pct,
    ROUND(100.0 * rows_with_provider_offer_pct / total_rows, 1) AS pct_with_provider_offer_pct,
    flagged_provider_over_3x,
    flagged_qpa_below_50pct,
    rows_with_rvu,
    ROUND(100.0 * rows_with_rvu / total_rows, 1) AS pct_with_rvu
FROM coverage

total_rows,total_disputes,total_dlis,rows_with_medicare_rate,pct_with_medicare_rate,rows_with_service_code,pct_with_service_code,rows_with_qpa_pct_medicare,pct_with_qpa_pct_medicare,rows_with_provider_offer_pct,pct_with_provider_offer_pct,flagged_provider_over_3x,flagged_qpa_below_50pct,rows_with_rvu,pct_with_rvu
2747354,1058514,2747354,2005361,73.0,2747354,100.0,1439136,52.4,1728981,62.9,8049,1434391,2005361,73.0


In [0]:
%sql
-- ============================================================
-- SIGNAL A — Medicare Multiple Analysis (Enhances Block 2)
-- ============================================================
-- provider_offer_as_pct_of_medicare is stored as a RATIO:
--   0.5 = 50% of Medicare, 1.0 = at Medicare, 3.0 = 3× Medicare
--
-- KEY FINDING: Most providers ask BELOW Medicare at the median.
-- The real signal is:
--   1. Who has the highest P90/P99 (tail exploitation on specific codes)?
--   2. What % of DLIs exceed key Medicare multiples (selective exploitation)?
--   3. How does the provider ask compare to the QPA depression level?
--
-- This reframes the story: QPA depression is the dominant phenomenon.
-- Providers filing disputes are often trying to reach Medicare levels.
-- The FEW who systematically exceed Medicare on specific codes are outliers.

WITH provider_medicare_profile AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        COUNT(DLI_Number) AS total_dlis,
        
        -- Medicare multiple distribution (values are ratios: 1.0 = at Medicare)
        COUNT(CASE WHEN provider_offer_as_pct_of_medicare IS NOT NULL THEN 1 END) AS dlis_with_medicare_comparison,
        ROUND(PERCENTILE(provider_offer_as_pct_of_medicare, 0.5), 3) AS median_offer_vs_medicare,
        ROUND(PERCENTILE(provider_offer_as_pct_of_medicare, 0.75), 3) AS p75_offer_vs_medicare,
        ROUND(PERCENTILE(provider_offer_as_pct_of_medicare, 0.9), 3) AS p90_offer_vs_medicare,
        ROUND(PERCENTILE(provider_offer_as_pct_of_medicare, 0.99), 3) AS p99_offer_vs_medicare,
        ROUND(AVG(provider_offer_as_pct_of_medicare), 3) AS avg_offer_vs_medicare,
        
        -- What fraction exceeds key Medicare multiples
        ROUND(100.0 * COUNT(CASE WHEN provider_offer_as_pct_of_medicare > 1.0 THEN 1 END)
            / NULLIF(COUNT(CASE WHEN provider_offer_as_pct_of_medicare IS NOT NULL THEN 1 END), 0), 1) AS pct_dlis_above_medicare,
        ROUND(100.0 * COUNT(CASE WHEN provider_offer_as_pct_of_medicare > 2.0 THEN 1 END)
            / NULLIF(COUNT(CASE WHEN provider_offer_as_pct_of_medicare IS NOT NULL THEN 1 END), 0), 1) AS pct_dlis_over_2x_medicare,
        ROUND(100.0 * COUNT(CASE WHEN flag_provider_over_3x_medicare = TRUE THEN 1 END)
            / NULLIF(COUNT(CASE WHEN provider_offer_as_pct_of_medicare IS NOT NULL THEN 1 END), 0), 1) AS pct_dlis_over_3x_medicare,
        
        -- QPA depression context
        ROUND(PERCENTILE(qpa_as_pct_of_medicare, 0.5), 3) AS median_qpa_vs_medicare,
        ROUND(100.0 * COUNT(CASE WHEN flag_qpa_below_50pct_medicare = TRUE THEN 1 END)
            / NULLIF(COUNT(CASE WHEN qpa_as_pct_of_medicare IS NOT NULL THEN 1 END), 0), 1) AS pct_dlis_qpa_below_50pct_medicare,
        
        -- Win rates for context
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%' AND Default_Decision != 'Yes'
            THEN Dispute_Number END) / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS merit_win_rate,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes' AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS default_win_rate
    FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
    WHERE Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
    HAVING COUNT(DISTINCT Dispute_Number) >= 100
)
SELECT
    Provider_Email_Domain,
    total_disputes,
    dlis_with_medicare_comparison,
    
    -- Provider offer vs Medicare (ratio: 1.0 = at Medicare)
    median_offer_vs_medicare,
    p75_offer_vs_medicare,
    p90_offer_vs_medicare,
    p99_offer_vs_medicare,
    pct_dlis_above_medicare,
    pct_dlis_over_2x_medicare,
    pct_dlis_over_3x_medicare,
    
    -- QPA depression context
    median_qpa_vs_medicare,
    pct_dlis_qpa_below_50pct_medicare,
    
    -- The spread: how much is the provider asking above QPA in Medicare terms
    ROUND(median_offer_vs_medicare / NULLIF(median_qpa_vs_medicare, 0), 2) AS provider_qpa_spread_ratio,
    
    -- Win context
    merit_win_rate,
    default_win_rate,
    
    -- Classification (ratio-corrected thresholds)
    CASE
        WHEN pct_dlis_over_3x_medicare > 10 THEN 'HIGH TAIL (>10% of DLIs exceed 3× Medicare)'
        WHEN pct_dlis_over_2x_medicare > 20 THEN 'ELEVATED TAIL (>20% exceed 2× Medicare)'
        WHEN pct_dlis_above_medicare > 50 THEN 'ABOVE MEDICARE MAJORITY'
        WHEN median_offer_vs_medicare > 1.0 THEN 'MEDIAN ABOVE MEDICARE'
        WHEN median_offer_vs_medicare > 0.5 THEN 'MODERATE (50-100% of Medicare)'
        ELSE 'BELOW HALF MEDICARE'
    END AS medicare_position_tier,
    
    CASE
        WHEN pct_dlis_over_3x_medicare > 10 OR pct_dlis_over_2x_medicare > 20
             OR median_offer_vs_medicare > 1.0
        THEN TRUE ELSE FALSE
    END AS flagged_above_medicare,
    
    CASE
        WHEN pct_dlis_over_3x_medicare > 10
        THEN CONCAT(pct_dlis_over_3x_medicare, '% of DLIs exceed 3× Medicare. P90=', p90_offer_vs_medicare, '×, P99=', p99_offer_vs_medicare, '×. Default rate: ', default_win_rate, '%')
        WHEN pct_dlis_over_2x_medicare > 20
        THEN CONCAT(pct_dlis_over_2x_medicare, '% exceed 2× Medicare. Median=', median_offer_vs_medicare, '×. QPA at ', median_qpa_vs_medicare, '× Medicare')
        WHEN median_offer_vs_medicare > 1.0
        THEN CONCAT('Median ask is ', median_offer_vs_medicare, '× Medicare (', pct_dlis_above_medicare, '% of DLIs above Medicare rate)')
        ELSE NULL
    END AS flag_reason
FROM provider_medicare_profile
WHERE dlis_with_medicare_comparison >= 50
ORDER BY pct_dlis_over_3x_medicare DESC, pct_dlis_above_medicare DESC, median_offer_vs_medicare DESC

Provider_Email_Domain,total_disputes,dlis_with_medicare_comparison,median_offer_vs_medicare,p75_offer_vs_medicare,p90_offer_vs_medicare,p99_offer_vs_medicare,pct_dlis_above_medicare,pct_dlis_over_2x_medicare,pct_dlis_over_3x_medicare,median_qpa_vs_medicare,pct_dlis_qpa_below_50pct_medicare,provider_qpa_spread_ratio,merit_win_rate,default_win_rate,medicare_position_tier,flagged_above_medicare,flag_reason
agilityrcm.com,306,311,0.073,0.445,4.491,45.503,22.2,18.3,12.5,0.004,100.0,18.25,57.2,13.7,HIGH TAIL (>10% of DLIs exceed 3× Medicare),true,"12.5% of DLIs exceed 3× Medicare. P90=4.491×, P99=45.503×. Default rate: 13.7%"
usrcm.net,844,1772,0.742,2.078,3.231,4.846,45.1,25.6,11.1,0.011,99.9,67.45,45.7,22.7,HIGH TAIL (>10% of DLIs exceed 3× Medicare),true,"11.1% of DLIs exceed 3× Medicare. P90=3.231×, P99=4.846×. Default rate: 22.7%"
totalcare.us,14006,25696,0.206,0.654,2.632,9.656,20.0,13.3,8.1,0.024,96.1,8.58,41.1,35.2,BELOW HALF MEDICARE,false,null
atlanticneurocare.com,190,130,0.137,0.251,0.293,7.193,5.4,5.4,5.4,0.002,100.0,68.5,71.6,12.6,BELOW HALF MEDICARE,false,null
civie.com,345,558,0.223,1.661,2.444,3.176,34.4,14.9,3.0,0.012,100.0,18.58,82.9,4.1,BELOW HALF MEDICARE,false,null
mdcapitaladvisors.com,20180,17664,0.183,0.5,1.176,6.479,11.9,5.4,2.8,0.003,99.7,61.0,77.1,13.6,BELOW HALF MEDICARE,false,null
integrityrcmcorp.com,217,579,0.119,0.461,0.694,6.178,7.3,2.9,2.8,0.001,100.0,119.0,49.8,20.3,BELOW HALF MEDICARE,false,null
collaborativeimaging.com,2200,3179,0.387,1.337,2.324,3.176,28.0,14.2,2.4,0.01,100.0,38.7,52.8,7.8,BELOW HALF MEDICARE,false,null
khcfirm.com,107,101,0.159,0.531,1.134,3.503,12.9,3.0,2.0,0.002,98.0,79.5,87.9,6.5,BELOW HALF MEDICARE,false,null
gottliebandgreenspan.com,15017,9501,0.139,0.412,1.078,4.12,10.9,3.8,2.0,0.003,99.8,46.33,69.7,18.8,BELOW HALF MEDICARE,false,null


In [0]:
%sql
-- ============================================================
-- SIGNAL C — Service Code Concentration (CPT-HHI)
-- ============================================================
-- Measures how concentrated each provider's filings are across
-- CPT/HCPCS service codes. Uses Herfindahl-Hirschman Index.
--
-- A legitimate clinical practice files across diverse codes.
-- A billing factory files 80%+ on 2-3 codes (e.g., 99285, 99284).
--
-- HHI interpretation: 0.0 = perfectly diverse, 1.0 = all one code
-- Flag: HHI > 0.25 (equivalent to filing >50% on one code)
--       AND top_3_code_share > 70%

WITH code_volume AS (
    SELECT
        Provider_Email_Domain,
        Service_Code,
        COUNT(DLI_Number) AS code_dli_count
    FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
    WHERE Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
      AND Service_Code IS NOT NULL
      AND TRIM(Service_Code) != ''
    GROUP BY Provider_Email_Domain, Service_Code
),
provider_totals AS (
    SELECT
        Provider_Email_Domain,
        SUM(code_dli_count) AS total_dlis,
        COUNT(DISTINCT Service_Code) AS distinct_codes
    FROM code_volume
    GROUP BY Provider_Email_Domain
),
code_shares AS (
    SELECT
        cv.Provider_Email_Domain,
        cv.Service_Code,
        cv.code_dli_count,
        pt.total_dlis,
        pt.distinct_codes,
        CAST(cv.code_dli_count AS DOUBLE) / pt.total_dlis AS code_share,
        ROW_NUMBER() OVER (PARTITION BY cv.Provider_Email_Domain ORDER BY cv.code_dli_count DESC) AS code_rank
    FROM code_volume cv
    JOIN provider_totals pt ON cv.Provider_Email_Domain = pt.Provider_Email_Domain
    WHERE pt.total_dlis >= 500  -- minimum volume for meaningful HHI
),
hhi_calc AS (
    SELECT
        Provider_Email_Domain,
        total_dlis,
        distinct_codes,
        ROUND(SUM(code_share * code_share), 4) AS cpt_hhi
    FROM code_shares
    GROUP BY Provider_Email_Domain, total_dlis, distinct_codes
),
top_codes AS (
    SELECT
        Provider_Email_Domain,
        MAX(CASE WHEN code_rank = 1 THEN Service_Code END) AS top1_code,
        MAX(CASE WHEN code_rank = 1 THEN ROUND(code_share * 100, 1) END) AS top1_share_pct,
        MAX(CASE WHEN code_rank = 2 THEN Service_Code END) AS top2_code,
        MAX(CASE WHEN code_rank = 2 THEN ROUND(code_share * 100, 1) END) AS top2_share_pct,
        MAX(CASE WHEN code_rank = 3 THEN Service_Code END) AS top3_code,
        MAX(CASE WHEN code_rank = 3 THEN ROUND(code_share * 100, 1) END) AS top3_share_pct,
        SUM(CASE WHEN code_rank <= 3 THEN ROUND(code_share * 100, 1) ELSE 0 END) AS top3_combined_share
    FROM code_shares
    WHERE code_rank <= 3
    GROUP BY Provider_Email_Domain
),
-- Join with dispute/win stats for context
win_stats AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes' AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS default_win_rate
    FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
    WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
)
SELECT
    h.Provider_Email_Domain,
    ws.total_disputes,
    h.total_dlis,
    h.distinct_codes,
    h.cpt_hhi,
    tc.top1_code,
    tc.top1_share_pct,
    tc.top2_code,
    tc.top2_share_pct,
    tc.top3_code,
    tc.top3_share_pct,
    tc.top3_combined_share,
    ws.default_win_rate,
    
    -- Classification
    CASE
        WHEN h.cpt_hhi > 0.50 THEN 'EXTREME CONCENTRATION (single-code factory)'
        WHEN h.cpt_hhi > 0.25 THEN 'HIGH CONCENTRATION (2-3 code dependency)'
        WHEN h.cpt_hhi > 0.10 THEN 'MODERATE CONCENTRATION'
        ELSE 'DIVERSIFIED'
    END AS concentration_tier,
    
    CASE
        WHEN h.cpt_hhi > 0.25 AND tc.top3_combined_share > 70
        THEN TRUE ELSE FALSE
    END AS flagged_cpt_concentration,
    
    CASE
        WHEN h.cpt_hhi > 0.50
        THEN CONCAT('HHI=', h.cpt_hhi, '. Single code dominance: ', tc.top1_code, ' at ', tc.top1_share_pct, '% of all DLIs. Only ', h.distinct_codes, ' distinct codes.')
        WHEN h.cpt_hhi > 0.25 AND tc.top3_combined_share > 70
        THEN CONCAT('HHI=', h.cpt_hhi, '. Top 3 codes = ', tc.top3_combined_share, '% of DLIs: ', tc.top1_code, '(', tc.top1_share_pct, '%), ', tc.top2_code, '(', tc.top2_share_pct, '%), ', tc.top3_code, '(', tc.top3_share_pct, '%)')
        ELSE NULL
    END AS flag_reason
FROM hhi_calc h
JOIN top_codes tc ON h.Provider_Email_Domain = tc.Provider_Email_Domain
JOIN win_stats ws ON h.Provider_Email_Domain = ws.Provider_Email_Domain
ORDER BY h.cpt_hhi DESC

Provider_Email_Domain,total_disputes,total_dlis,distinct_codes,cpt_hhi,top1_code,top1_share_pct,top2_code,top2_share_pct,top3_code,top3_share_pct,top3_combined_share,default_win_rate,concentration_tier,flagged_cpt_concentration,flag_reason
natera.com,1866,2925,10,0.6123,81420,77.6,81329,6.5,81220,5.3,89.39999999999999,20.2,EXTREME CONCENTRATION (single-code factory),true,HHI=0.6123. Single code dominance: 81420 at 77.6% of all DLIs. Only 10 distinct codes.
clearchoiceer.com,531,531,3,0.5892,99284,71.4,99285,28.2,99291,0.4,100.00000000000001,45.4,EXTREME CONCENTRATION (single-code factory),true,HHI=0.5892. Single code dominance: 99284 at 71.4% of all DLIs. Only 3 distinct codes.
expresserbilling.com,91,549,16,0.5141,99284,67.4,99285,24.2,99283,2.6,94.2,31.9,EXTREME CONCENTRATION (single-code factory),true,HHI=0.5141. Single code dominance: 99284 at 67.4% of all DLIs. Only 16 distinct codes.
ecp.net,277,858,4,0.488,99284,60.8,99285,34.1,99291,2.6,97.5,12.6,HIGH CONCENTRATION (2-3 code dependency),true,"HHI=0.488. Top 3 codes = 97.5% of DLIs: 99284(60.8%), 99285(34.1%), 99291(2.6%)"
ventrahealth.com,1282,38200,18,0.4842,99285,50.3,99284,48.1,99283,1.0,99.4,16.2,HIGH CONCENTRATION (2-3 code dependency),true,"HHI=0.4842. Top 3 codes = 99.4% of DLIs: 99285(50.3%), 99284(48.1%), 99283(1.0%)"
apollomd.com,1035,18876,10,0.4786,99284,55.6,99285,41.1,99291,1.9,98.60000000000001,17.2,HIGH CONCENTRATION (2-3 code dependency),true,"HHI=0.4786. Top 3 codes = 98.60000000000001% of DLIs: 99284(55.6%), 99285(41.1%), 99291(1.9%)"
scphealth.com,2511,3273,6,0.4326,99284,52.3,99285,39.4,99291,4.6,96.29999999999998,25.4,HIGH CONCENTRATION (2-3 code dependency),true,"HHI=0.4326. Top 3 codes = 96.29999999999998% of DLIs: 99284(52.3%), 99285(39.4%), 99291(4.6%)"
agshealth.com,50422,73767,20,0.4317,99284,52.6,99285,38.9,99291,4.6,96.1,21.3,HIGH CONCENTRATION (2-3 code dependency),true,"HHI=0.4317. Top 3 codes = 96.1% of DLIs: 99284(52.6%), 99285(38.9%), 99291(4.6%)"
scp-health.com,47223,67024,15,0.4311,99284,52.8,99285,38.5,99291,4.9,96.2,22.4,HIGH CONCENTRATION (2-3 code dependency),true,"HHI=0.4311. Top 3 codes = 96.2% of DLIs: 99284(52.8%), 99285(38.5%), 99291(4.9%)"
victoriaemergency.com,47,2723,6,0.4179,99284,48.1,99285,42.7,99283,5.6,96.4,0.0,HIGH CONCENTRATION (2-3 code dependency),true,"HHI=0.4179. Top 3 codes = 96.4% of DLIs: 99284(48.1%), 99285(42.7%), 99283(5.6%)"


In [0]:
%sql
-- ============================================================
-- SIGNAL B — Clinical Complexity Profile (RVU Analysis)
-- ============================================================
-- Are flagged providers disproportionately filing on LOW-complexity
-- services? Low work_rvu = simple service. High volume on simple
-- services + high default rate = industrial billing operation.
--
-- total_non_facility_rvu incorporates:
--   work_rvu (physician time/skill) + PE_rvu (practice expense) + malpractice_rvu
-- Higher RVU = more complex/resource-intensive service.
--
-- Signal: providers with BELOW-MEDIAN avg RVU who also have high default rates
-- are exploiting simple, formulaic services where plans may be less likely to respond.

WITH provider_rvu AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        COUNT(DLI_Number) AS total_dlis,
        COUNT(CASE WHEN total_non_facility_rvu IS NOT NULL THEN 1 END) AS dlis_with_rvu,
        
        -- RVU distribution per provider
        ROUND(AVG(CAST(total_non_facility_rvu AS DOUBLE)), 2) AS avg_rvu,
        ROUND(PERCENTILE(CAST(total_non_facility_rvu AS DOUBLE), 0.5), 2) AS median_rvu,
        ROUND(PERCENTILE(CAST(total_non_facility_rvu AS DOUBLE), 0.25), 2) AS p25_rvu,
        ROUND(PERCENTILE(CAST(total_non_facility_rvu AS DOUBLE), 0.75), 2) AS p75_rvu,
        
        -- Distinct service complexity categories
        COUNT(DISTINCT CASE WHEN CAST(total_non_facility_rvu AS DOUBLE) < 2.0 THEN Service_Code END) AS low_complexity_codes,
        COUNT(DISTINCT CASE WHEN CAST(total_non_facility_rvu AS DOUBLE) >= 2.0 AND CAST(total_non_facility_rvu AS DOUBLE) < 5.0 THEN Service_Code END) AS mid_complexity_codes,
        COUNT(DISTINCT CASE WHEN CAST(total_non_facility_rvu AS DOUBLE) >= 5.0 THEN Service_Code END) AS high_complexity_codes,
        
        -- Volume share by complexity band
        ROUND(100.0 * COUNT(CASE WHEN CAST(total_non_facility_rvu AS DOUBLE) < 2.0 THEN 1 END)
            / NULLIF(COUNT(CASE WHEN total_non_facility_rvu IS NOT NULL THEN 1 END), 0), 1) AS pct_dlis_low_rvu,
        ROUND(100.0 * COUNT(CASE WHEN CAST(total_non_facility_rvu AS DOUBLE) >= 2.0 AND CAST(total_non_facility_rvu AS DOUBLE) < 5.0 THEN 1 END)
            / NULLIF(COUNT(CASE WHEN total_non_facility_rvu IS NOT NULL THEN 1 END), 0), 1) AS pct_dlis_mid_rvu,
        ROUND(100.0 * COUNT(CASE WHEN CAST(total_non_facility_rvu AS DOUBLE) >= 5.0 THEN 1 END)
            / NULLIF(COUNT(CASE WHEN total_non_facility_rvu IS NOT NULL THEN 1 END), 0), 1) AS pct_dlis_high_rvu,
        
        -- Win rates
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%' AND Default_Decision != 'Yes'
            THEN Dispute_Number END) / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS merit_win_rate,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes' AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS default_win_rate
    FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
    WHERE Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
    HAVING COUNT(DISTINCT Dispute_Number) >= 100
       AND COUNT(CASE WHEN total_non_facility_rvu IS NOT NULL THEN 1 END) >= 50
)
SELECT
    Provider_Email_Domain,
    total_disputes,
    dlis_with_rvu,
    avg_rvu,
    median_rvu,
    p25_rvu,
    p75_rvu,
    pct_dlis_low_rvu,
    pct_dlis_mid_rvu,
    pct_dlis_high_rvu,
    low_complexity_codes,
    mid_complexity_codes,
    high_complexity_codes,
    merit_win_rate,
    default_win_rate,
    
    -- Classification
    CASE
        WHEN median_rvu < 2.0 AND pct_dlis_low_rvu > 70 THEN 'LOW COMPLEXITY DOMINANT'
        WHEN median_rvu < 3.0 AND pct_dlis_low_rvu > 50 THEN 'MODERATE-LOW COMPLEXITY'
        WHEN median_rvu >= 5.0 THEN 'HIGH COMPLEXITY'
        ELSE 'BALANCED'
    END AS complexity_tier,
    
    -- Flag: low RVU + high default rate = industrial billing on simple services
    CASE
        WHEN median_rvu < 3.0 AND default_win_rate > 25 AND total_disputes > 1000
        THEN TRUE ELSE FALSE
    END AS flagged_low_rvu_high_default,
    
    CASE
        WHEN median_rvu < 2.0 AND default_win_rate > 30 AND total_disputes > 1000
        THEN CONCAT('Low-complexity factory: median RVU=', median_rvu, ', ', pct_dlis_low_rvu, '% low-RVU DLIs, ', default_win_rate, '% default rate on ', total_disputes, ' disputes')
        WHEN median_rvu < 3.0 AND default_win_rate > 25 AND total_disputes > 1000
        THEN CONCAT('Moderate-low complexity: median RVU=', median_rvu, ', default rate ', default_win_rate, '% on ', total_disputes, ' disputes')
        ELSE NULL
    END AS flag_reason
FROM provider_rvu
ORDER BY 
    CASE WHEN median_rvu < 3.0 AND default_win_rate > 25 AND total_disputes > 1000 THEN 0 ELSE 1 END,
    default_win_rate DESC

Provider_Email_Domain,total_disputes,dlis_with_rvu,avg_rvu,median_rvu,p25_rvu,p75_rvu,pct_dlis_low_rvu,pct_dlis_mid_rvu,pct_dlis_high_rvu,low_complexity_codes,mid_complexity_codes,high_complexity_codes,merit_win_rate,default_win_rate,complexity_tier,flagged_low_rvu_high_default,flag_reason
altushealthsystem.com,9379,4994,2.75,2.11,0.93,3.6,37.3,42.7,20.0,67,35,25,30.5,50.8,BALANCED,true,"Moderate-low complexity: median RVU=2.11, default rate 50.8% on 9379 disputes"
bellaireer.com,1069,1393,2.45,2.11,1.01,3.6,37.6,52.5,9.8,38,26,10,38.6,48.6,BALANCED,true,"Moderate-low complexity: median RVU=2.11, default rate 48.6% on 1069 disputes"
legacyhealthllc.com,2064,1393,2.51,1.79,0.44,3.9,53.6,23.1,23.3,38,26,15,18.1,47.4,MODERATE-LOW COMPLEXITY,true,"Low-complexity factory: median RVU=1.79, 53.6% low-RVU DLIs, 47.4% default rate on 2064 disputes"
fam-llc.com,38045,98289,2.79,2.6,0.82,3.6,43.9,38.2,17.9,132,84,145,39.1,43.1,BALANCED,true,"Moderate-low complexity: median RVU=2.6, default rate 43.1% on 38045 disputes"
totalcare.us,14006,37650,2.33,2.11,0.44,3.6,46.3,43.6,10.2,98,69,50,41.1,35.2,BALANCED,true,"Moderate-low complexity: median RVU=2.11, default rate 35.2% on 14006 disputes"
primehealthcare.com,16470,43162,2.66,1.22,0.44,3.6,53.9,25.5,20.7,185,152,388,58.4,32.8,MODERATE-LOW COMPLEXITY,true,"Low-complexity factory: median RVU=1.22, 53.9% low-RVU DLIs, 32.8% default rate on 16470 disputes"
redrockrad.com,1088,8266,3.68,2.6,0.9,4.83,47.1,28.0,24.9,63,42,101,69.9,27.4,BALANCED,true,"Moderate-low complexity: median RVU=2.6, default rate 27.4% on 1088 disputes"
chs.net,114,112,3.4,3.6,2.11,3.6,3.6,80.4,16.1,1,4,4,0.9,96.5,BALANCED,false,null
heightsrcm.com,436,662,3.77,3.6,1.29,5.22,26.6,32.8,40.6,25,19,9,35.8,53.4,BALANCED,false,null
gryphonhc.com,3450,4337,5.02,5.22,3.6,5.22,5.3,31.9,62.7,27,32,29,39.4,51.6,HIGH COMPLEXITY,false,null


In [0]:
%sql
-- ============================================================
-- SIGNAL D — QPA Depression & Legitimacy Segmentation
-- ============================================================
-- Uses flag_qpa_below_50pct_medicare to segment providers into:
--   1. "Depressed QPA environment" filers (plan offers far below Medicare)
--   2. "Normal QPA environment" filers (plan offers are reasonable)
--
-- A provider whose disputes are ALL in depressed-QPA environments
-- may have legitimate grievances. A provider with mixed environments
-- who STILL defaults heavily in normal-QPA areas is more suspect.
--
-- Additionally: computes the provider's default rate CONDITIONED ON
-- QPA depression status. If default_rate is high EVEN when QPA is
-- fair (>50% Medicare), that’s a stronger gaming signal.

WITH provider_qpa_env AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        
        -- QPA depression coverage
        COUNT(CASE WHEN qpa_as_pct_of_medicare IS NOT NULL THEN 1 END) AS dlis_with_qpa_context,
        COUNT(CASE WHEN flag_qpa_below_50pct_medicare = TRUE THEN 1 END) AS dlis_qpa_depressed,
        COUNT(CASE WHEN qpa_as_pct_of_medicare IS NOT NULL AND flag_qpa_below_50pct_medicare = FALSE THEN 1 END) AS dlis_qpa_fair,
        
        -- Rate of QPA depression
        ROUND(100.0 * COUNT(CASE WHEN flag_qpa_below_50pct_medicare = TRUE THEN 1 END)
            / NULLIF(COUNT(CASE WHEN qpa_as_pct_of_medicare IS NOT NULL THEN 1 END), 0), 1) AS pct_dlis_qpa_depressed,
        
        -- Default rate in DEPRESSED QPA environment
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN flag_qpa_below_50pct_medicare = TRUE
             AND Default_Decision = 'Yes'
             AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END)
            / NULLIF(COUNT(DISTINCT CASE WHEN flag_qpa_below_50pct_medicare = TRUE THEN Dispute_Number END), 0), 1) AS default_rate_when_qpa_depressed,
        
        -- Default rate in FAIR QPA environment
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN qpa_as_pct_of_medicare IS NOT NULL
             AND flag_qpa_below_50pct_medicare = FALSE
             AND Default_Decision = 'Yes'
             AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END)
            / NULLIF(COUNT(DISTINCT CASE
                WHEN qpa_as_pct_of_medicare IS NOT NULL AND flag_qpa_below_50pct_medicare = FALSE
                THEN Dispute_Number END), 0), 1) AS default_rate_when_qpa_fair,
        
        -- Overall default rate for reference
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes' AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS overall_default_rate,
        
        -- Merit rate
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%' AND Default_Decision != 'Yes'
            THEN Dispute_Number END) / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS merit_win_rate
    FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
    WHERE Provider_Email_Domain IS NOT NULL
      AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
    HAVING COUNT(DISTINCT Dispute_Number) >= 500
       AND COUNT(CASE WHEN qpa_as_pct_of_medicare IS NOT NULL THEN 1 END) >= 100
)
SELECT
    Provider_Email_Domain,
    total_disputes,
    dlis_with_qpa_context,
    pct_dlis_qpa_depressed,
    default_rate_when_qpa_depressed,
    default_rate_when_qpa_fair,
    overall_default_rate,
    merit_win_rate,
    
    -- The spread: gaming even in fair-QPA environments
    ROUND(default_rate_when_qpa_fair - default_rate_when_qpa_depressed, 1) AS fair_vs_depressed_default_spread,
    
    -- Legitimacy classification
    CASE
        WHEN pct_dlis_qpa_depressed > 90 AND overall_default_rate < 20
        THEN 'LEGITIMATE GRIEVANCE (all depressed QPA, low defaults)'
        WHEN pct_dlis_qpa_depressed > 90 AND overall_default_rate >= 20
        THEN 'DEPRESSED QPA + HIGH DEFAULT (mixed signal)'
        WHEN default_rate_when_qpa_fair > 30 AND total_disputes > 1000
        THEN 'GAMING SIGNAL (high defaults even when QPA is fair)'
        WHEN default_rate_when_qpa_fair > 20 AND total_disputes > 5000
        THEN 'ELEVATED GAMING (moderate defaults in fair-QPA environment)'
        ELSE 'UNCLEAR / INSUFFICIENT DATA'
    END AS legitimacy_tier,
    
    CASE
        WHEN default_rate_when_qpa_fair > 30 AND total_disputes > 1000
        THEN TRUE
        WHEN default_rate_when_qpa_fair > 20 AND total_disputes > 5000
        THEN TRUE
        ELSE FALSE
    END AS flagged_gaming_in_fair_qpa,
    
    CASE
        WHEN default_rate_when_qpa_fair > 30 AND total_disputes > 1000
        THEN CONCAT('Default rate ', default_rate_when_qpa_fair, '% EVEN when QPA is fair (>50% Medicare). Overall: ', overall_default_rate, '%. Gaming persists regardless of QPA environment.')
        WHEN default_rate_when_qpa_fair > 20 AND total_disputes > 5000
        THEN CONCAT('Default rate ', default_rate_when_qpa_fair, '% in fair-QPA environment on ', total_disputes, ' disputes. Not purely a QPA grievance issue.')
        ELSE NULL
    END AS flag_reason
FROM provider_qpa_env
ORDER BY 
    CASE WHEN default_rate_when_qpa_fair > 30 AND total_disputes > 1000 THEN 0
         WHEN default_rate_when_qpa_fair > 20 AND total_disputes > 5000 THEN 1
         ELSE 2 END,
    default_rate_when_qpa_fair DESC NULLS LAST

Provider_Email_Domain,total_disputes,dlis_with_qpa_context,pct_dlis_qpa_depressed,default_rate_when_qpa_depressed,default_rate_when_qpa_fair,overall_default_rate,merit_win_rate,fair_vs_depressed_default_spread,legitimacy_tier,flagged_gaming_in_fair_qpa,flag_reason
summit-az.com,10511,736,99.7,3.0,50.0,9.5,79.9,47.0,"LEGITIMATE GRIEVANCE (all depressed QPA, low defaults)",true,Default rate 50.0% EVEN when QPA is fair (>50% Medicare). Overall: 9.5%. Gaming persists regardless of QPA environment.
tenethealth.com,783,816,99.8,7.2,50.0,23.1,52.0,42.8,DEPRESSED QPA + HIGH DEFAULT (mixed signal),false,null
saintfrancis.com,598,754,98.9,35.3,25.0,43.3,53.2,-10.3,DEPRESSED QPA + HIGH DEFAULT (mixed signal),false,null
envisionhealth.com,25008,27287,100.0,3.5,16.7,14.7,70.8,13.2,"LEGITIMATE GRIEVANCE (all depressed QPA, low defaults)",false,null
roundtmc.com,8545,15061,99.8,10.5,12.5,37.2,45.8,2.0,DEPRESSED QPA + HIGH DEFAULT (mixed signal),false,null
adventhealth.com,911,2307,94.2,4.9,10.8,15.0,21.7,5.9,"LEGITIMATE GRIEVANCE (all depressed QPA, low defaults)",false,null
memorialvillageer.com,1894,1572,94.0,5.4,10.7,31.0,51.3,5.3,DEPRESSED QPA + HIGH DEFAULT (mixed signal),false,null
gottliebandgreenspan.com,15017,8445,99.8,4.0,10.0,18.8,69.7,6.0,"LEGITIMATE GRIEVANCE (all depressed QPA, low defaults)",false,null
aimbillingsolutions.com,6323,2149,99.4,5.1,9.1,46.1,36.0,4.0,DEPRESSED QPA + HIGH DEFAULT (mixed signal),false,null
erevenuebilling.com,6210,1671,97.5,3.3,8.6,16.5,56.4,5.3,"LEGITIMATE GRIEVANCE (all depressed QPA, low defaults)",false,null


In [0]:
%sql
-- ============================================================
-- ENHANCED BLOCK 9 — Final Composite Audit Ranking (Medicare-Enhanced)
-- ============================================================
-- Incorporates original 8 block flags PLUS 2 new Medicare-anchored signals:
--   b9:  CPT Concentration (HHI > 0.25 AND disputes > 1000)
--   b10: Low-RVU + High Default (median RVU < 3.0 AND default > 25%)
--
-- Source table: fee_schedule_joined_oon_emergency_nonemergency
-- Tier 1: 6+ flags | Tier 2: 4-5 | Monitoring: 1-3

WITH base_stats AS (
    SELECT
        Provider_Email_Domain,
        COUNT(DISTINCT Dispute_Number) AS total_disputes,
        COUNT(DLI_Number) AS total_dlis,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Payment_Determination_Outcome ILIKE '%provider%' AND Default_Decision != 'Yes'
            THEN Dispute_Number END) / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS merit_win_rate,
        ROUND(100.0 * COUNT(DISTINCT CASE
            WHEN Default_Decision = 'Yes' AND Payment_Determination_Outcome ILIKE '%provider%'
            THEN Dispute_Number END) / NULLIF(COUNT(DISTINCT Dispute_Number), 0), 1) AS default_win_rate
    FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
    WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
    GROUP BY Provider_Email_Domain
),
national AS (
    SELECT COUNT(DISTINCT Dispute_Number) AS nat FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
),
b1_flags AS (
    SELECT DISTINCT Provider_Email_Domain, 1 AS b1
    FROM base_stats CROSS JOIN national
    WHERE 100.0 * total_disputes / nat > 1.0 OR total_disputes > 5000
),
b2_flags AS (
    SELECT Provider_Email_Domain, 1 AS b2 FROM (
        SELECT Provider_Email_Domain,
               PERCENTILE(TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE), 0.5) AS med
        FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
        WHERE Payment_Determination_Outcome ILIKE '%provider%' AND Default_Decision != 'Yes'
          AND Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
          AND TRY_CAST(NULLIF(TRIM(Prevailing_Party_Offer_as_of_QPA), '+') AS DOUBLE) IS NOT NULL
        GROUP BY Provider_Email_Domain
    ) t WHERE med > 30 AND NOT isnan(med)
),
b3_flags AS (
    SELECT DISTINCT Provider_Email_Domain, 1 AS b3 FROM (
        SELECT t.Provider_Email_Domain, t.Health_Plan_Issuer_Name,
               COUNT(DISTINCT t.Dispute_Number) AS filed,
               COUNT(DISTINCT CASE WHEN t.Default_Decision='Yes' AND t.Payment_Determination_Outcome ILIKE '%provider%' THEN t.Dispute_Number END) AS dw
        FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency t
        WHERE t.Provider_Email_Domain IS NOT NULL AND TRIM(t.Provider_Email_Domain) != ''
        GROUP BY t.Provider_Email_Domain, t.Health_Plan_Issuer_Name
        HAVING COUNT(DISTINCT t.Dispute_Number) >= 25
    ) pp WHERE 100.0 * dw / NULLIF(filed, 0) > 40
),
b4_flags AS (
    SELECT Provider_Email_Domain, 1 AS b4 FROM (
        SELECT Provider_Email_Domain,
               SUM(CASE WHEN LOWER(data_quarter) LIKE '%q1%' THEN 1 ELSE 0 END) AS q1c,
               SUM(CASE WHEN LOWER(data_quarter) LIKE '%q2%' THEN 1 ELSE 0 END) AS q2c
        FROM (SELECT DISTINCT Provider_Email_Domain, Dispute_Number, data_quarter
              FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
              WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != '') sub
        GROUP BY Provider_Email_Domain
    ) q WHERE q1c > 0 AND 100.0 * (q2c - q1c) / q1c > 25
),
b5_flags AS (
    SELECT t.Provider_Email_Domain, 1 AS b5 FROM (
        SELECT Provider_Email_Domain, LOWER(TRIM(Type_of_Dispute)) AS dt,
               COUNT(DISTINCT Dispute_Number) AS dc,
               COUNT(DISTINCT CASE WHEN Default_Decision='Yes' AND Payment_Determination_Outcome ILIKE '%provider%' THEN Dispute_Number END) AS dw
        FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
        WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
        GROUP BY Provider_Email_Domain, LOWER(TRIM(Type_of_Dispute))
    ) t
    JOIN (
        SELECT Provider_Email_Domain, LOWER(TRIM(Type_of_Dispute)) AS dt,
               COUNT(DISTINCT Dispute_Number) AS dc,
               COUNT(DISTINCT CASE WHEN Default_Decision='Yes' AND Payment_Determination_Outcome ILIKE '%provider%' THEN Dispute_Number END) AS dw
        FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
        WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
        GROUP BY Provider_Email_Domain, LOWER(TRIM(Type_of_Dispute))
    ) s ON t.Provider_Email_Domain = s.Provider_Email_Domain AND s.dt = 'single'
    WHERE t.dt = 'batched' AND t.dc >= 50
      AND 100.0 * t.dw / NULLIF(t.dc, 0) > 100.0 * s.dw / NULLIF(s.dc, 0) + 10
),
b6_flags AS (
    SELECT DISTINCT Provider_Email_Domain, 1 AS b6 FROM (
        SELECT Provider_Email_Domain, Health_Plan_Issuer_Name,
               COUNT(DISTINCT CASE WHEN Default_Decision='Yes' AND Payment_Determination_Outcome ILIKE '%provider%' THEN Dispute_Number END) AS dw,
               SUM(COUNT(DISTINCT CASE WHEN Default_Decision='Yes' AND Payment_Determination_Outcome ILIKE '%provider%' THEN Dispute_Number END))
                   OVER (PARTITION BY Health_Plan_Issuer_Name) AS plan_total
        FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
        WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
        GROUP BY Provider_Email_Domain, Health_Plan_Issuer_Name
    ) pp WHERE 100.0 * dw / NULLIF(plan_total, 0) > 20 AND dw > 50
),
b7_flags AS (
    SELECT bs.Provider_Email_Domain, 1 AS b7
    FROM base_stats bs
    WHERE bs.default_win_rate > 20 AND bs.total_disputes > 5000
),
b8_flags AS (
    SELECT Provider_Email_Domain, 1 AS b8 FROM (
        SELECT Provider_Email_Domain,
               SUM(CASE WHEN LOWER(data_quarter) LIKE '%q1%' THEN 1 ELSE 0 END) AS q1c,
               SUM(CASE WHEN LOWER(data_quarter) LIKE '%q2%' THEN 1 ELSE 0 END) AS q2c
        FROM (SELECT DISTINCT Provider_Email_Domain, Dispute_Number, data_quarter
              FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
              WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != '') sub
        GROUP BY Provider_Email_Domain
        HAVING SUM(CASE WHEN LOWER(data_quarter) LIKE '%q1%' THEN 1 ELSE 0 END) >= 1000
           AND SUM(CASE WHEN LOWER(data_quarter) LIKE '%q2%' THEN 1 ELSE 0 END) >= 1000
    ) q WHERE q1c > 0
),
-- NEW FLAGS
b9_flags AS (
    SELECT h.Provider_Email_Domain, 1 AS b9 FROM (
        SELECT cv.Provider_Email_Domain,
               SUM(POWER(CAST(cv.cnt AS DOUBLE) / pt.total, 2)) AS hhi
        FROM (SELECT Provider_Email_Domain, Service_Code, COUNT(*) AS cnt
              FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
              WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
                AND Service_Code IS NOT NULL AND TRIM(Service_Code) != ''
              GROUP BY Provider_Email_Domain, Service_Code) cv
        JOIN (SELECT Provider_Email_Domain, COUNT(*) AS total
              FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
              WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
                AND Service_Code IS NOT NULL AND TRIM(Service_Code) != ''
              GROUP BY Provider_Email_Domain HAVING COUNT(*) >= 500) pt
            ON cv.Provider_Email_Domain = pt.Provider_Email_Domain
        GROUP BY cv.Provider_Email_Domain
    ) h
    JOIN base_stats bs ON h.Provider_Email_Domain = bs.Provider_Email_Domain
    WHERE h.hhi > 0.25 AND bs.total_disputes > 1000
),
b10_flags AS (
    SELECT r.Provider_Email_Domain, 1 AS b10 FROM (
        SELECT Provider_Email_Domain,
               PERCENTILE(CAST(total_non_facility_rvu AS DOUBLE), 0.5) AS med_rvu
        FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
        WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
          AND total_non_facility_rvu IS NOT NULL
        GROUP BY Provider_Email_Domain
    ) r
    JOIN base_stats bs ON r.Provider_Email_Domain = bs.Provider_Email_Domain
    WHERE r.med_rvu < 3.0 AND bs.default_win_rate > 25 AND bs.total_disputes > 1000
),
-- HHI for score multiplier
hhi_values AS (
    SELECT cv.Provider_Email_Domain,
           ROUND(SUM(POWER(CAST(cv.cnt AS DOUBLE) / pt.total, 2)), 4) AS cpt_hhi
    FROM (SELECT Provider_Email_Domain, Service_Code, COUNT(*) AS cnt
          FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
          WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
            AND Service_Code IS NOT NULL AND TRIM(Service_Code) != ''
          GROUP BY Provider_Email_Domain, Service_Code) cv
    JOIN (SELECT Provider_Email_Domain, COUNT(*) AS total
          FROM idre.idre_silver.fee_schedule_joined_oon_emergency_nonemergency
          WHERE Provider_Email_Domain IS NOT NULL AND TRIM(Provider_Email_Domain) != ''
            AND Service_Code IS NOT NULL AND TRIM(Service_Code) != ''
          GROUP BY Provider_Email_Domain HAVING COUNT(*) >= 100) pt
        ON cv.Provider_Email_Domain = pt.Provider_Email_Domain
    GROUP BY cv.Provider_Email_Domain
),
combined AS (
    SELECT
        bs.Provider_Email_Domain,
        bs.total_disputes,
        bs.total_dlis,
        bs.merit_win_rate,
        bs.default_win_rate,
        COALESCE(b1.b1, 0) AS b1,
        COALESCE(b2.b2, 0) AS b2,
        COALESCE(b3.b3, 0) AS b3,
        COALESCE(b4.b4, 0) AS b4,
        COALESCE(b5.b5, 0) AS b5,
        COALESCE(b6.b6, 0) AS b6,
        COALESCE(b7.b7, 0) AS b7,
        COALESCE(b8.b8, 0) AS b8,
        COALESCE(b9.b9, 0) AS b9,
        COALESCE(b10.b10, 0) AS b10,
        COALESCE(hv.cpt_hhi, 0) AS cpt_hhi
    FROM base_stats bs
    LEFT JOIN b1_flags b1 ON bs.Provider_Email_Domain = b1.Provider_Email_Domain
    LEFT JOIN b2_flags b2 ON bs.Provider_Email_Domain = b2.Provider_Email_Domain
    LEFT JOIN b3_flags b3 ON bs.Provider_Email_Domain = b3.Provider_Email_Domain
    LEFT JOIN b4_flags b4 ON bs.Provider_Email_Domain = b4.Provider_Email_Domain
    LEFT JOIN b5_flags b5 ON bs.Provider_Email_Domain = b5.Provider_Email_Domain
    LEFT JOIN b6_flags b6 ON bs.Provider_Email_Domain = b6.Provider_Email_Domain
    LEFT JOIN b7_flags b7 ON bs.Provider_Email_Domain = b7.Provider_Email_Domain
    LEFT JOIN b8_flags b8 ON bs.Provider_Email_Domain = b8.Provider_Email_Domain
    LEFT JOIN b9_flags b9 ON bs.Provider_Email_Domain = b9.Provider_Email_Domain
    LEFT JOIN b10_flags b10 ON bs.Provider_Email_Domain = b10.Provider_Email_Domain
    LEFT JOIN hhi_values hv ON bs.Provider_Email_Domain = hv.Provider_Email_Domain
)
SELECT
    Provider_Email_Domain,
    total_disputes,
    total_dlis,
    merit_win_rate,
    default_win_rate,
    b1, b2, b3, b4, b5, b6, b7, b8, b9, b10,
    (b1+b2+b3+b4+b5+b6+b7+b8+b9+b10) AS total_flags,
    cpt_hhi,
    ROUND(
        (b1+b2+b3+b4+b5+b6+b7+b8+b9+b10)
        * LOG10(GREATEST(total_disputes, 10))
        * (1 + default_win_rate/100.0)
        * (1 + 0.3 * cpt_hhi)
    , 1) AS risk_score_enhanced,
    CASE
        WHEN (b1+b2+b3+b4+b5+b6+b7+b8+b9+b10) >= 6 THEN 'Tier 1 — Audit Referral'
        WHEN (b1+b2+b3+b4+b5+b6+b7+b8+b9+b10) >= 4 THEN 'Tier 2 — Enhanced Monitoring'
        WHEN (b1+b2+b3+b4+b5+b6+b7+b8+b9+b10) >= 1 THEN 'Monitoring'
        ELSE 'Clear'
    END AS tier
FROM combined
WHERE (b1+b2+b3+b4+b5+b6+b7+b8+b9+b10) >= 1
ORDER BY risk_score_enhanced DESC

Provider_Email_Domain,total_disputes,total_dlis,merit_win_rate,default_win_rate,b1,b2,b3,b4,b5,b6,b7,b8,b9,b10,total_flags,cpt_hhi,risk_score_enhanced,tier
fam-llc.com,38045,242035,39.1,43.1,1,0,1,1,1,1,1,1,0,1,8,0.0221,52.8,Tier 1 — Audit Referral
qmacsmso.com,30581,32569,41.0,39.5,1,0,1,1,0,1,1,1,1,0,7,0.353,48.4,Tier 1 — Audit Referral
totalcare.us,14006,97644,41.1,35.2,1,0,1,1,1,1,1,1,0,1,8,0.0211,45.1,Tier 1 — Audit Referral
halomd.com,180706,341867,61.8,25.5,1,0,1,1,0,1,1,1,0,0,6,0.0501,40.2,Tier 1 — Audit Referral
primehealthcare.com,16470,147268,58.4,32.8,1,0,1,1,0,1,1,1,0,1,7,0.0169,39.4,Tier 1 — Audit Referral
scp-health.com,47223,67024,67.3,22.4,1,0,1,0,0,1,1,1,1,0,6,0.4311,38.8,Tier 1 — Audit Referral
agshealth.com,50422,73767,65.1,21.3,1,0,1,0,0,1,1,1,1,0,6,0.4317,38.7,Tier 1 — Audit Referral
usap.com,23846,34813,67.1,20.9,1,0,1,1,1,1,1,1,0,0,7,0.0276,37.4,Tier 1 — Audit Referral
teamhealth.com,168799,304189,87.0,7.7,1,0,1,1,0,1,0,1,1,0,6,0.3286,37.1,Tier 1 — Audit Referral
altushealthsystem.com,9379,26079,30.5,50.8,1,0,1,0,0,1,1,1,0,1,6,0.0142,36.1,Tier 1 — Audit Referral
